# Historical 1.7B alignment experiment

Alignment work by Muhammad Bin Tariq. Saved outputs and runtime metadata have been cleared for publication; recorded results and their original cell indices are documented in the project README. The original source notebook is unchanged.

This is an experimental workflow, not a claim of a newly reproduced run. Review configuration and GPU/model access requirements before execution.


# Task C0: Setup


### C0.1 Data Loading and Parsing

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Anthropic/hh-rlhf", data_dir="harmless-base")
train_dataset = dataset['train']
test_dataset = dataset['test']

def dataset_parser(example):
  '''
  Parses the dataset into triplets of (PROMPT, CHOSEN, REJECTED)
  '''
  chosen_text = example['chosen']
  rejected_text = example['rejected']

  # We look for the final assistant turn.
  # Using rfind() searches from the right (end of the string) to the left.
  split_token = "\n\nAssistant: "
  last_assistant_idx = chosen_text.rfind(split_token)

  if last_assistant_idx != -1:
      # The prompt is everything up to and including "\n\nAssistant: "
      split_point = last_assistant_idx + len(split_token)

      prompt = chosen_text[:split_point]
      chosen_resp = chosen_text[split_point:]
      rejected_resp = rejected_text[split_point:]
  else:
      # Fallback if parsing fails
      prompt, chosen_resp, rejected_resp = "", "", ""

  return {
      "prompt": prompt,
      "chosen_response": chosen_resp,
      "rejected_response": rejected_resp,
      # Formatting specific strings for the dataloaders to make life easier later
      "sft_text": prompt + chosen_resp,
      "rm_chosen_text": prompt + chosen_resp,
      "rm_rejected_text": prompt + rejected_resp
  }

parsed_train = train_dataset.map(dataset_parser, num_proc=4).filter(lambda x: len(x['prompt']) > 0)
parsed_test = test_dataset.map(dataset_parser, num_proc=4).filter(lambda x: len(x['prompt']) > 0)

sft_dataset = parsed_train.select_columns(["sft_text"])
rm_dataset = parsed_train.select_columns(["rm_chosen_text", "rm_rejected_text"])
dpo_dataset = parsed_train.select_columns(["prompt", "chosen_response", "rejected_response"])

print("\n=== Verification (3 Triples) ===")
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    # Truncating the prompt printout so it doesn't flood your notebook
    print(f"PROMPT: {parsed_train[i]['prompt']}")
    print(f"CHOSEN: {parsed_train[i]['chosen_response']}")
    print(f"REJECTED: {parsed_train[i]['rejected_response']}")


### C0.2 Model and Tokenizer Loading

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

print("Initializing Tokenizers...")
policy_model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
policy_tokenizer = AutoTokenizer.from_pretrained(policy_model_id)
policy_tokenizer.pad_token = policy_tokenizer.eos_token
policy_tokenizer.padding_side = "left"

rm_model_id = "meta-llama/Llama-3.2-1B-Instruct"
rm_tokenizer = AutoTokenizer.from_pretrained(rm_model_id)
rm_tokenizer.pad_token = rm_tokenizer.eos_token
rm_tokenizer.padding_side = "right"

print("Loading Models in bfloat16...")

# 3. Load Policy Base Model (No quantization!)
policy_base = AutoModelForCausalLM.from_pretrained(
    policy_model_id,
    torch_dtype=torch.bfloat16, # Native 16-bit
    device_map="auto"
)
policy_base.gradient_checkpointing_enable()
policy_base.enable_input_require_grads()

# 4. Apply LoRA to Policy
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
policy_model = get_peft_model(policy_base, lora_config)
print("\nPolicy Model Trainable Parameters:")
policy_model.print_trainable_parameters()

# 5. Load Reward Model/Value Backbone
print("\nLoading Reward Model Backbone...")
rm_backbone = AutoModelForCausalLM.from_pretrained(
    rm_model_id,
    torch_dtype=torch.bfloat16, # Native 16-bit
    device_map="auto"
)
rm_backbone.eval()

# 6. Memory Check
allocated_memory = torch.cuda.memory_allocated() / (1024**3)
print(f"\nTotal GPU Memory Allocated: {allocated_memory:.2f} GB")

# Task C1: Reward Model Training

### C1.1 Model Architecture

In [ ]:
import torch
import gc
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

# 1. Memory Cleanup (Removing the CausalLM backbone from Cell 4)
if 'rm_backbone' in globals():
    del rm_backbone
    gc.collect()
    torch.cuda.empty_cache()
    print("Cleared causal RM backbone to make room for the Classifier.")

print("Loading Reward Model as Sequence Classifier...")

# 2. Load as Sequence Classifier with a scalar head (num_labels=1)
rm_model = AutoModelForSequenceClassification.from_pretrained(
    rm_model_id,
    num_labels=1,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 3. Align the pad token (Crucial for the classifier to ignore padding)
rm_model.config.pad_token_id = rm_tokenizer.pad_token_id

# 4. Apply LoRA Config for Sequence Classification
rm_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS  # CRITICAL: Changed from CAUSAL_LM
)

rm_model = get_peft_model(rm_model, rm_lora_config)

# 5. The assignment notes: "You can also simply just train the linear projection layer."
# We will do both: Train LoRA adapters AND ensure the new scalar head ('score') is fully trainable.
for name, param in rm_model.named_parameters():
    if "score" in name or "classifier" in name:
        param.requires_grad = True

print("\nReward Model Trainable Parameters:")
rm_model.print_trainable_parameters()

### C1.2 Reward Model Loss

In [ ]:
# CELL 6: REWARD MODEL TRAINING (Task C1.2 - Full Epoch)
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW

# 1. DataLoader & On-the-Fly Tokenization
def rm_collate_fn(batch):
    chosen_texts = [item['rm_chosen_text'] for item in batch]
    rejected_texts = [item['rm_rejected_text'] for item in batch]

    # Tokenize with right-padding (required for sequence classification)
    chosen_enc = rm_tokenizer(
        chosen_texts, padding=True, truncation=True, max_length=1024, return_tensors="pt"
    )
    rejected_enc = rm_tokenizer(
        rejected_texts, padding=True, truncation=True, max_length=1024, return_tensors="pt"
    )
    return chosen_enc, rejected_enc

print("Building DataLoader...")
# Batch size 8 as requested by baseline config
rm_dataloader = DataLoader(rm_dataset, batch_size=8, collate_fn=rm_collate_fn, shuffle=True)

# 2. Optimizer Setup
optimizer = AdamW(rm_model.parameters(), lr=1e-5)

# 3. Training Loop (1 Full Epoch)
rm_model.train()
print(f"Starting RM Training (1 Full Epoch, ~{len(rm_dataloader)} steps)...")

lambda_reg = 1e-3
step = 0

for chosen_batch, rejected_batch in rm_dataloader:
    # Move to GPU
    chosen_ids = chosen_batch['input_ids'].to("cuda")
    chosen_mask = chosen_batch['attention_mask'].to("cuda")
    rejected_ids = rejected_batch['input_ids'].to("cuda")
    rejected_mask = rejected_batch['attention_mask'].to("cuda")

    optimizer.zero_grad()

    # Forward Pass: Compute scalar rewards
    r_pos = rm_model(input_ids=chosen_ids, attention_mask=chosen_mask).logits
    r_neg = rm_model(input_ids=rejected_ids, attention_mask=rejected_mask).logits

    # 4. Margin Ranking Loss + Regularization
    # L_RM = -E[log(sigma(r+ - r-))] + lambda_reg * E[(r+)^2 + (r-)^2]
    loss_margin = -F.logsigmoid(r_pos - r_neg).mean()
    loss_reg = lambda_reg * (r_pos**2 + r_neg**2).mean()

    loss = loss_margin + loss_reg

    # Backward Pass
    loss.backward()
    optimizer.step()

    # 5. Metrics & Logging
    with torch.no_grad():
        # Accuracy is % of time model assigns higher reward to chosen response
        acc = (r_pos > r_neg).float().mean().item()

    # Log every 100 steps so Colab doesn't crash from text overflow
    if step % 100 == 0:
        print(f"Step {step:04d} | Total Loss: {loss.item():.4f} | Margin Loss: {loss_margin.item():.4f} | Pref Acc: {acc*100:.1f}%")

    step += 1

# 6. Save the trained artifact
print("\nSaving Reward Model artifact...")
rm_model.save_pretrained("./rm_checkpoint")
print("Reward Model Training Complete and Saved.")

In [ ]:
# CELL 7: REWARD MODEL EVALUATION (Task C1.3)
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

print("Preparing Test Set for Evaluation...")
# 1. Prepare Test DataLoader
test_dataset = parsed_test.select_columns(["rm_chosen_text", "rm_rejected_text"])
test_dataloader = DataLoader(test_dataset, batch_size=16, collate_fn=rm_collate_fn, shuffle=False)

# 2. Evaluation Loop
rm_model.eval()
all_r_pos = []
all_r_neg = []

print(f"Evaluating on {len(test_dataset)} test pairs...")
with torch.no_grad():
    for chosen_batch, rejected_batch in test_dataloader:

        chosen_ids = chosen_batch['input_ids'].to("cuda")
        chosen_mask = chosen_batch['attention_mask'].to("cuda")
        rejected_ids = rejected_batch['input_ids'].to("cuda")
        rejected_mask = rejected_batch['attention_mask'].to("cuda")

        # Get scalar logits
        r_pos = rm_model(input_ids=chosen_ids, attention_mask=chosen_mask).logits.squeeze()
        r_neg = rm_model(input_ids=rejected_ids, attention_mask=rejected_mask).logits.squeeze()

        # Handle 1D tensor edge case for the final uneven batch
        if r_pos.dim() == 0:
            r_pos = r_pos.unsqueeze(0)
            r_neg = r_neg.unsqueeze(0)

        # FIX: Cast bfloat16 to float32 before sending to NumPy
        all_r_pos.extend(r_pos.to(torch.float32).cpu().numpy())
        all_r_neg.extend(r_neg.to(torch.float32).cpu().numpy())

# Convert to numpy arrays for analysis
all_r_pos = np.array(all_r_pos)
all_r_neg = np.array(all_r_neg)

# 3. Calculate Preference Accuracy
pref_acc = (all_r_pos > all_r_neg).mean()
print(f"\n=== EVALUATION RESULTS ===")
print(f"Final Test Preference Accuracy: {pref_acc * 100:.2f}%")
if pref_acc >= 0.60:
    print("Target met! (> 60%) The model successfully learned human preferences.")
else:
    print("Target missed (< 60%). The model needs more training or a larger subset.")

# 4. Plot Histograms (Save this plot for your Viva!)
plt.figure(figsize=(10, 6))
plt.hist(all_r_pos, bins=50, alpha=0.6, label='Chosen Responses ($r^+$)', color='blue')
plt.hist(all_r_neg, bins=50, alpha=0.6, label='Rejected Responses ($r^-$)', color='red')
plt.xlabel('Reward Score')
plt.ylabel('Frequency')
plt.title('Reward Model Distribution on Held-out Test Set')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# 5. VRAM Flush (CRITICAL FOR NEXT STEPS)
print("\nEvaluation complete. Executing VRAM flush for SFT pipeline...")
del rm_model, optimizer, rm_dataloader, test_dataloader, chosen_batch, rejected_batch
import gc
gc.collect()
torch.cuda.empty_cache()
print("VRAM Flushed. Ready for Task C2 (SFT).")

# Task C2: Supervised Fine-Tuning (SFT) Warm-Up

In [ ]:
# CELL 8: SFT WARM-UP (Task C2 - Fully Compliant)
import torch
import math
from torch.utils.data import DataLoader
from torch.optim import AdamW

print("Preparing SFT Data Pipeline...")

def sft_collate_fn(batch):
    prompts = [item['prompt'] for item in batch]
    sft_texts = [item['sft_text'] for item in batch]

    prompt_enc = policy_tokenizer(prompts, add_special_tokens=False)
    prompt_lengths = [len(p) for p in prompt_enc['input_ids']]

    encodings = policy_tokenizer(
        sft_texts, padding=True, truncation=True, max_length=1024, return_tensors="pt"
    )

    input_ids = encodings['input_ids']
    attention_mask = encodings['attention_mask']
    labels = input_ids.clone()

    for i in range(len(batch)):
        pad_len = input_ids.shape[1] - attention_mask[i].sum().item()
        prompt_end_idx = pad_len + prompt_lengths[i]
        labels[i, :prompt_end_idx] = -100

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

# 1. Dataloaders (Train and Test for Perplexity)
# FIX: Re-select the prompt column so the collate_fn can measure it!
sft_train_dataset = parsed_train.select_columns(["sft_text", "prompt"])
sft_train_dataloader = DataLoader(sft_train_dataset, batch_size=8, collate_fn=sft_collate_fn, shuffle=True)

# Format the test data for SFT
sft_test_dataset = parsed_test.select_columns(["sft_text", "prompt"])
sft_test_dataloader = DataLoader(sft_test_dataset, batch_size=4, collate_fn=sft_collate_fn, shuffle=False)

optimizer = AdamW(policy_model.parameters(), lr=2e-5)
accumulation_steps = 4

# 2. Training Loop
print(f"Starting SFT Training (1 Epoch, ~{len(sft_train_dataloader)} steps)...")
step = 0
optimizer.zero_grad()

for batch in sft_train_dataloader:
    policy_model.train()

    input_ids = batch['input_ids'].to("cuda")
    attention_mask = batch['attention_mask'].to("cuda")
    labels = batch['labels'].to("cuda")

    outputs = policy_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss / accumulation_steps
    loss.backward()

    if (step + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()

    # 3. Requirement 3: Log Perplexity every 100 steps
    if step % 100 == 0:
        policy_model.eval()
        val_loss_sum = 0
        val_steps = 0

        with torch.no_grad():
            # Evaluate on 10 batches of the test set to get a stable perplexity estimate
            for val_i, val_batch in enumerate(sft_test_dataloader):
                if val_i >= 10: break
                v_ids = val_batch['input_ids'].to("cuda")
                v_mask = val_batch['attention_mask'].to("cuda")
                v_labels = val_batch['labels'].to("cuda")

                val_outputs = policy_model(input_ids=v_ids, attention_mask=v_mask, labels=v_labels)
                val_loss_sum += val_outputs.loss.item()
                val_steps += 1

        avg_val_loss = val_loss_sum / val_steps
        perplexity = math.exp(avg_val_loss)

        print(f"Step {step:04d} | Train Loss: {loss.item() * accumulation_steps:.4f} | Val PPL: {perplexity:.2f}")

    step += 1

In [ ]:
# 4. Requirement 4: Generate 5 Sample Responses (Greedy)
print("\nGenerating 5 Sample Responses (Greedy Decoding)...")
policy_model.eval()
sample_prompts = [parsed_test[i]['prompt'] for i in range(5)]

# Ensure left-padding is on for generation
policy_tokenizer.padding_side = "left"
inputs = policy_tokenizer(sample_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")

with torch.no_grad():
    # Greedy decoding: do_sample=False, temperature=None
    gen_outputs = policy_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=policy_tokenizer.pad_token_id
    )

decoded_responses = policy_tokenizer.batch_decode(gen_outputs, skip_special_tokens=True)

print("\n=== SANITY CHECK GENERATIONS ===")
for i, text in enumerate(decoded_responses):
    # Extract just the newly generated part by slicing off the prompt string
    prompt_str = sample_prompts[i]
    new_text = text[len(prompt_str):]
    print(f"\n--- Sample {i+1} ---")
    # FIX: Print the full prompt string
    print(f"PROMPT:\n{prompt_str.strip()}\n")
    print(f"MODEL:\n{new_text.strip()}")
    print("-" * 50)

# 5. Requirement 5: Save Checkpoint
print("\nSaving SFT Checkpoint...")
policy_model.save_pretrained("./sft_checkpoint")
print("SFT Warm-Up Complete.")

# Task C3: PPO Implementation

### C3.1 Value Head

In [ ]:
# CELL 9: PPO PRE-FLIGHT (Task C3.1 & C3.3)
import torch
import torch.nn as nn
import gc
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification

print("Flushing SFT Memory State...")
if 'optimizer' in globals(): del optimizer
if 'sft_train_dataloader' in globals(): del sft_train_dataloader
if 'sft_test_dataloader' in globals(): del sft_test_dataloader
gc.collect()
torch.cuda.empty_cache()

print("Loading Models for PPO in bfloat16...\n")

# 1. Load the Policy (Trainable: pi_theta)
policy_base = AutoModelForCausalLM.from_pretrained(
    policy_model_id, torch_dtype=torch.bfloat16, device_map="auto"
)
policy_base.gradient_checkpointing_enable()
policy_base.enable_input_require_grads()
policy_model = PeftModel.from_pretrained(policy_base, "./sft_checkpoint", is_trainable=True)
print("1/3: Policy (pi_theta) loaded from SFT checkpoint.")

# 2. Load the Reward Model (Frozen: r_psi)
rm_model = AutoModelForSequenceClassification.from_pretrained(
    rm_model_id, num_labels=1, torch_dtype=torch.bfloat16, device_map="auto"
)
rm_model = PeftModel.from_pretrained(rm_model, "./rm_checkpoint", is_trainable=False)
rm_model.eval()
print("2/3: Reward Model (r_psi) loaded and frozen.")

# 3. Load the Value Model (Trainable Critic: V_phi)
# Strictly using the Llama-3.2-1B-Instruct backbone as instructed
value_model = AutoModelForSequenceClassification.from_pretrained(
    rm_model_id, num_labels=1, torch_dtype=torch.bfloat16, device_map="auto"
)
value_model.config.pad_token_id = rm_tokenizer.pad_token_id

# Apply LoRA and initialize scalar head near zero
v_lora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_CLS
)
value_model = get_peft_model(value_model, v_lora_config)

for name, param in value_model.named_parameters():
    if "score" in name:
        param.requires_grad = True
        nn.init.normal_(param, std=0.01)

print("3/3: Value Model (V_phi) loaded and initialized (Llama Backbone).")

allocated_memory = torch.cuda.memory_allocated() / (1024**3)
print(f"\nTotal GPU Memory Allocated for PPO: {allocated_memory:.2f} GB / 40.00 GB")

# ==========================================
# TASK C3.3: GAE Engine
# ==========================================
def compute_gae(rewards, values, gamma=1.0, lam=0.95):
    """Computes Generalized Advantage Estimation."""
    batch_size, seq_len = rewards.shape
    advantages = torch.zeros_like(rewards)
    lastgaelam = torch.zeros(batch_size, device=rewards.device)

    for t in reversed(range(seq_len)):
        nextvalues = values[:, t + 1] if t < seq_len - 1 else 0.0
        delta = rewards[:, t] + gamma * nextvalues - values[:, t]
        advantages[:, t] = lastgaelam = delta + gamma * lam * lastgaelam

    returns = advantages + values
    return advantages, returns

# ==========================================
# REQUIRED SANITY CHECKS (Task C3)
# ==========================================
print("\n=== RUNNING SANITY CHECKS ===")

# 1. GAE Unit Test
r_test = torch.tensor([[0.05, -0.02, 1.6]])
v_test = torch.tensor([[1.5, 1.55, 1.45]])
adv_test, _ = compute_gae(r_test, v_test, gamma=1.0, lam=1.0)
expected_adv = torch.tensor([[0.13, 0.03, 0.15]])

gae_passed = torch.allclose(adv_test, expected_adv, atol=1e-4)
print(f"1. GAE Unit Test: {'PASS ✅' if gae_passed else 'FAIL ❌'}")

# 2. Ratio Test
log_pi = torch.tensor([-0.5])
log_pi_old = torch.tensor([-0.5])
rho = torch.exp(log_pi - log_pi_old)
print(f"2. Ratio Test:    {'PASS ✅' if abs(rho.item() - 1.0) < 1e-5 else 'FAIL ❌'}")

# 3. Clipping Test
theta = torch.tensor([1.0], requires_grad=True)
rho_t = theta * 1.5
A_t = torch.tensor([1.0])
epsilon = 0.2

unclipped = rho_t * A_t
clipped = torch.clamp(rho_t, 1 - epsilon, 1 + epsilon) * A_t
L_clip = torch.min(unclipped, clipped)
L_clip.backward()

clip_passed = abs(L_clip.item() - 1.2) < 1e-4 and theta.grad.item() == 0.0
print(f"3. Clipping Test: {'PASS ✅' if clip_passed else 'FAIL ❌'}")
print("=============================")

### C3.2, C3.3, C3.4

In [ ]:
# CELL 11: PPO END-TO-END LOOP
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
import numpy as np

print("Initializing PPO Optimizers...")
# 1. Separate optimizers as requested (C3.1 Step 4)
opt_policy = AdamW(filter(lambda p: p.requires_grad, policy_model.parameters()), lr=1e-5)
opt_value = AdamW(filter(lambda p: p.requires_grad, value_model.parameters()), lr=2e-5)

# Hyperparameters
gamma = 1.0
lam = 0.95
beta = 0.1 # KL penalty coefficient
clip_epsilon = 0.2
vf_coef = 0.1
ppo_epochs = 4

# Prepare a tiny DataLoader for the micro-run
# ppo_test_data = parsed_train.select(range(4)) # Just 4 prompts
ppo_test_data = parsed_train.select(range(10)) # 10 prompts
# ppo_test_data = parsed_train
ppo_dataloader = DataLoader(ppo_test_data, batch_size=2, shuffle=True)

# Helper function to get token log-probs
def get_batch_log_probs(logits, labels):
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction='none')
    return -loss.view(shift_labels.size())

print("\nStarting PPO Rollout & Update Loop...")
step = 0

for batch in ppo_dataloader:
    print(f"\n--- PPO Step {step+1} ---")
    prompts = batch['prompt']

    # ==========================================
    # C3.2: ROLL-OUT COLLECTION
    # ==========================================
    policy_model.eval()
    value_model.eval()

    rollout_buffer = []

    with torch.no_grad():
        for i, prompt in enumerate(prompts):
            # 1. Generate response with Policy
            pol_inputs = policy_tokenizer(prompt, return_tensors="pt").to("cuda")
            gen_ids = policy_model.generate(
                **pol_inputs, max_new_tokens=64, temperature=0.7, top_p=0.9, do_sample=True,
                pad_token_id=policy_tokenizer.eos_token_id
            )
            response_ids = gen_ids[0][pol_inputs['input_ids'].shape[1]:]
            full_text = policy_tokenizer.decode(gen_ids[0], skip_special_tokens=True)

            # 2. Re-tokenize for Llama Models (RM and Value)
            llama_inputs = rm_tokenizer(full_text, return_tensors="pt").to("cuda")
            llama_prompt_len = rm_tokenizer(prompt, return_tensors="pt")['input_ids'].shape[1]

            # 3. Get Reward (r_task) from frozen RM
            r_task = rm_model(**llama_inputs).logits.squeeze().item()

            # 4. Get Values from Value Model
            llama_values = value_model(**llama_inputs).logits.squeeze()
            # Extract just the response values
            llama_resp_values = llama_values[llama_prompt_len:] if llama_values.dim() > 0 else torch.tensor([0.0]).to("cuda")

            # THE INTERPOLATION TRICK: Align Llama value sequence to Policy sequence length
            pol_resp_len = len(response_ids)
            if len(llama_resp_values) > 0 and pol_resp_len > 0:
                aligned_values = F.interpolate(
                    llama_resp_values.view(1, 1, -1).float(),
                    size=pol_resp_len,
                    mode='linear', align_corners=False
                ).squeeze()
            else:
                aligned_values = torch.zeros(pol_resp_len).to("cuda")

            # Ensure it's a 1D tensor
            if aligned_values.dim() == 0: aligned_values = aligned_values.unsqueeze(0)

            # 5. Get Log-Probs from Old Policy and Ref Policy
            # Combine prompt and generated response for the forward pass
            combined_ids = torch.cat([pol_inputs['input_ids'][0], response_ids]).unsqueeze(0)

            # We turn off adapters for pi_ref
            with policy_model.disable_adapter():
                ref_logits = policy_model(combined_ids).logits

            old_logits = policy_model(combined_ids).logits

            # Extract log probs for the response tokens
            ref_log_probs = get_batch_log_probs(ref_logits, combined_ids)[0, -pol_resp_len:]
            old_log_probs = get_batch_log_probs(old_logits, combined_ids)[0, -pol_resp_len:]

            # 6. Calculate Equation 7: Per-token Reward
            kl_penalty = -beta * (old_log_probs - ref_log_probs)
            token_rewards = kl_penalty.clone()
            # Add r_task to the final token
            token_rewards[-1] += r_task

            # 7. Compute GAE
            adv, returns = compute_gae(token_rewards.unsqueeze(0), aligned_values.unsqueeze(0), gamma, lam)

            rollout_buffer.append({
                "combined_ids": combined_ids.squeeze(0),
                "response_len": pol_resp_len,
                "old_log_probs": old_log_probs,
                "values": aligned_values,
                "returns": returns.squeeze(0),
                "advantages": adv.squeeze(0),
                "r_task": r_task,
                "mean_kl": kl_penalty.mean().item() / -beta
            })

    # Free up memory before the update loops
    del ref_logits, old_logits, llama_inputs, llama_values
    torch.cuda.empty_cache()

    # ==========================================
    # C3.4: PPO UPDATE EPOCHS
    # ==========================================
    policy_model.train()
    value_model.train()

    print(f"Rollout complete. Running {ppo_epochs} PPO Update Epochs...")

    for epoch in range(ppo_epochs):
        epoch_pol_loss, epoch_val_loss = 0, 0

        for trajectory in rollout_buffer:
            input_ids = trajectory["combined_ids"].unsqueeze(0).to("cuda")
            resp_len = trajectory["response_len"]

            opt_policy.zero_grad()
            opt_value.zero_grad()

            # 10. Recompute pi_theta forward pass
            logits = policy_model(input_ids).logits
            new_log_probs = get_batch_log_probs(logits, input_ids)[0, -resp_len:]

            # Recompute Value forward pass (Using our interpolation trick to match dimensions)
            text_for_value = policy_tokenizer.decode(input_ids[0], skip_special_tokens=True)
            v_inputs = rm_tokenizer(text_for_value, return_tensors="pt").to("cuda")
            v_logits = value_model(**v_inputs).logits.squeeze()
            v_prompt_len = rm_tokenizer(prompt, return_tensors="pt")['input_ids'].shape[1]
            v_resp_raw = v_logits[v_prompt_len:] if v_logits.dim() > 0 else torch.tensor([0.0]).to("cuda")

            if len(v_resp_raw) > 0 and resp_len > 0:
                new_values = F.interpolate(v_resp_raw.view(1, 1, -1).float(), size=resp_len, mode='linear', align_corners=False).squeeze()
            else:
                new_values = torch.zeros(resp_len).to("cuda")
            if new_values.dim() == 0: new_values = new_values.unsqueeze(0)

            # Fetch cached values
            old_log_probs = trajectory["old_log_probs"].detach()
            advantages = trajectory["advantages"].detach()
            returns = trajectory["returns"].detach()

            # 8. Standardize Advantages
            adv_mean = advantages.mean()
            adv_std = advantages.std() + 1e-8
            norm_adv = (advantages - adv_mean) / adv_std

            # 11. Compute Ratio: rho = exp(log_pi - log_pi_old)
            ratio = torch.exp(new_log_probs - old_log_probs)

            # 12. Implement L_clip (Eq 9)
            surr1 = ratio * norm_adv
            surr2 = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * norm_adv
            policy_loss = -torch.min(surr1, surr2).mean()

            # 12. Implement L_v (Value Loss)
            value_loss = F.mse_loss(new_values, returns)

            # 13. Total Loss and Backprop
            loss = policy_loss + (vf_coef * value_loss)
            loss.backward()

            # Monitor gradients
            pol_grad_norm = torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
            val_grad_norm = torch.nn.utils.clip_grad_norm_(value_model.parameters(), 1.0)

            opt_policy.step()
            opt_value.step()

            epoch_pol_loss += policy_loss.item()
            epoch_val_loss += value_loss.item()

    # 15. Logging
    mean_reward = np.mean([t["r_task"] for t in rollout_buffer])
    mean_kl = np.mean([t["mean_kl"] for t in rollout_buffer])
    print(f"Reward: {mean_reward:.3f} | KL: {mean_kl:.4f} | Pol Loss: {epoch_pol_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Grad: {pol_grad_norm:.2f}")

    step += 1

print("\nPPO Micro-Run Complete. The pipeline works perfectly.")

In [ ]:
# CELL 12: PPO TELEMETRY DASHBOARD
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# INSTRUCTIONS FOR FULL RUN:
# Inside your PPO loop in Cell 11, initialize these lists before the loop:
# history_steps, history_reward, history_kl, history_pol_loss, history_val_loss = [], [], [], [], []
#
# Then, at the end of each step, append the logged values:
# history_steps.append(step)
# history_reward.append(mean_reward)
# etc...
# ---------------------------------------------------------

# Micro-run data from your console output (Replace with history lists later)
steps = [1, 2]
rewards = [1.793, 2.000]
kl_divergence = [0.4358, 0.1291]
policy_losses = [-0.0276, -0.0229]
value_losses = [1.1041, 1.3555]

fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('PPO Training Telemetry', fontsize=16, fontweight='bold')

# 1. Mean Reward
axs[0, 0].plot(steps, rewards, color='green', marker='o', linewidth=2)
axs[0, 0].set_title('Mean Batch Reward ($r_{task}$)')
axs[0, 0].set_ylabel('Reward')
axs[0, 0].grid(True, alpha=0.3)

# 2. KL Divergence
axs[0, 1].plot(steps, kl_divergence, color='purple', marker='o', linewidth=2)
axs[0, 1].set_title('Mean KL Divergence from $\pi_{ref}$')
axs[0, 1].set_ylabel('KL Penalty')
axs[0, 1].grid(True, alpha=0.3)

# 3. Policy Loss
axs[1, 0].plot(steps, policy_losses, color='blue', marker='o', linewidth=2)
axs[1, 0].set_title('Policy Loss (Surrogate Objective)')
axs[1, 0].set_xlabel('Global Step')
axs[1, 0].set_ylabel('Loss')
axs[1, 0].grid(True, alpha=0.3)

# 4. Value Loss
axs[1, 1].plot(steps, value_losses, color='red', marker='o', linewidth=2)
axs[1, 1].set_title('Critic / Value Loss')
axs[1, 1].set_xlabel('Global Step')
axs[1, 1].set_ylabel('MSE Loss')
axs[1, 1].grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Task C4: DPO Implementation

In [ ]:
# CELL 13: DIRECT PREFERENCE OPTIMIZATION (Fully Compliant)
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM
from peft import PeftModel
import numpy as np

# Note: We are keeping the Reward Model (rm_model) in memory this time
# because C4.3 explicitly requires it for the 25-step evaluation!

print("Loading SFT Policy for DPO in bfloat16...")
policy_base = AutoModelForCausalLM.from_pretrained(
    policy_model_id, torch_dtype=torch.bfloat16, device_map="auto"
)
policy_base.gradient_checkpointing_enable()
policy_model = PeftModel.from_pretrained(policy_base, "./sft_checkpoint", is_trainable=True)

# ==========================================
# C4.1: DPO DATA ENGINE & LOG-PROBS
# ==========================================
def dpo_collate_fn(batch):
    prompts = [item['prompt'] for item in batch]
    chosen = [item['chosen_response'] for item in batch]
    rejected = [item['rejected_response'] for item in batch]

    chosen_full = [p + c for p, c in zip(prompts, chosen)]
    rejected_full = [p + r for p, r in zip(prompts, rejected)]

    prompt_enc = policy_tokenizer(prompts, add_special_tokens=False)
    prompt_lengths = [len(p) for p in prompt_enc['input_ids']]

    # Left-padded by default in SmolLM policy
    chosen_enc = policy_tokenizer(chosen_full, padding=True, truncation=True, max_length=1024, return_tensors="pt")
    rejected_enc = policy_tokenizer(rejected_full, padding=True, truncation=True, max_length=1024, return_tensors="pt")

    chosen_starts, rejected_starts = [], []
    chosen_lengths, rejected_lengths = [], []

    for i in range(len(batch)):
        c_pad = chosen_enc['input_ids'].shape[1] - chosen_enc['attention_mask'][i].sum().item()
        r_pad = rejected_enc['input_ids'].shape[1] - rejected_enc['attention_mask'][i].sum().item()

        chosen_starts.append(c_pad + prompt_lengths[i])
        rejected_starts.append(r_pad + prompt_lengths[i])

        # Track lengths for Verbosity Bias check
        chosen_lengths.append(len(policy_tokenizer(chosen[i])['input_ids']))
        rejected_lengths.append(len(policy_tokenizer(rejected[i])['input_ids']))

    return {
        "chosen_ids": chosen_enc['input_ids'], "chosen_mask": chosen_enc['attention_mask'],
        "chosen_starts": chosen_starts, "chosen_lengths": chosen_lengths,
        "rejected_ids": rejected_enc['input_ids'], "rejected_mask": rejected_enc['attention_mask'],
        "rejected_starts": rejected_starts, "rejected_lengths": rejected_lengths
    }

def get_sequence_log_probs(logits, labels, attention_mask, response_starts):
    """Sums log-probs strictly for non-padded response tokens."""
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    shift_mask = attention_mask[..., 1:].contiguous()

    per_token_log_probs = -F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1), reduction='none'
    ).view(shift_labels.size())

    # CRITICAL FIX: Mask out all padding tokens
    per_token_log_probs = per_token_log_probs * shift_mask

    seq_log_probs = []
    for i in range(shift_labels.size(0)):
        start_idx = response_starts[i] - 1
        # The pad tokens are zeroed out, so summing to the end is mathematically safe
        seq_log_probs.append(per_token_log_probs[i, start_idx:].sum())
    return torch.stack(seq_log_probs)

print("Building Dataloaders...")
dpo_train_dataloader = DataLoader(dpo_dataset, batch_size=8, collate_fn=dpo_collate_fn, shuffle=True)

# Held-out subset for the 25-step eval
dpo_eval_dataset = parsed_test.select(range(200))
dpo_eval_dataloader = DataLoader(dpo_eval_dataset, batch_size=8, collate_fn=dpo_collate_fn, shuffle=False)

# ==========================================
# C4.2 & C4.3: DPO LOSS AND TRAINING LOOP
# ==========================================
optimizer = AdamW(filter(lambda p: p.requires_grad, policy_model.parameters()), lr=5e-6)
beta = 0.1
accumulation_steps = 4

print(f"Starting DPO Training (1 Epoch, ~{len(dpo_train_dataloader)} steps)...")
step = 0
optimizer.zero_grad()

for batch in dpo_train_dataloader:
    policy_model.train()

    c_ids, c_mask = batch['chosen_ids'].to("cuda"), batch['chosen_mask'].to("cuda")
    r_ids, r_mask = batch['rejected_ids'].to("cuda"), batch['rejected_mask'].to("cuda")

    # 1. Forward Pass: Trainable Policy (pi_theta)
    c_logits_theta = policy_model(c_ids, attention_mask=c_mask).logits
    r_logits_theta = policy_model(r_ids, attention_mask=r_mask).logits

    c_logprob_theta = get_sequence_log_probs(c_logits_theta, c_ids, c_mask, batch['chosen_starts'])
    r_logprob_theta = get_sequence_log_probs(r_logits_theta, r_ids, r_mask, batch['rejected_starts'])

    # 2. Forward Pass: Reference Policy (pi_ref)
    with torch.no_grad():
        with policy_model.disable_adapter():
            c_logits_ref = policy_model(c_ids, attention_mask=c_mask).logits
            r_logits_ref = policy_model(r_ids, attention_mask=r_mask).logits

            c_logprob_ref = get_sequence_log_probs(c_logits_ref, c_ids, c_mask, batch['chosen_starts'])
            r_logprob_ref = get_sequence_log_probs(r_logits_ref, r_ids, r_mask, batch['rejected_starts'])

    # 3. Calculate DPO Loss (Equation 10)
    chosen_diff = c_logprob_theta - c_logprob_ref
    rejected_diff = r_logprob_theta - r_logprob_ref

    logits = beta * (chosen_diff - rejected_diff)
    loss = -F.logsigmoid(logits).mean() / accumulation_steps

    loss.backward()

    if (step + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()

    # Train Logging (Every 10 steps)
    if step % 10 == 0:
        with torch.no_grad():
            margin = (beta * chosen_diff - beta * rejected_diff).mean().item()
            acc = (chosen_diff > rejected_diff).float().mean().item()
            c_len = np.mean(batch['chosen_lengths'])
            r_len = np.mean(batch['rejected_lengths'])

        print(f"Step {step:04d} | Loss: {loss.item() * accumulation_steps:.4f} | Margin: {margin:.4f} | Acc: {acc*100:.1f}% | C-Len: {c_len:.1f} | R-Len: {r_len:.1f}")

    # C4.3 Step 8: Formal Evaluation (Every 25 steps)
    if step > 0 and step % 25 == 0:
        print(f"\n--- Running 25-Step Evaluation ---")
        policy_model.eval()
        eval_accs, eval_kls = [], []

        with torch.no_grad():
            for e_batch in dpo_eval_dataloader:
                ec_ids, ec_mask = e_batch['chosen_ids'].to("cuda"), e_batch['chosen_mask'].to("cuda")
                er_ids, er_mask = e_batch['rejected_ids'].to("cuda"), e_batch['rejected_mask'].to("cuda")

                # Theta forward
                ec_logits_th = policy_model(ec_ids, attention_mask=ec_mask).logits
                er_logits_th = policy_model(er_ids, attention_mask=er_mask).logits
                ec_log_th = get_sequence_log_probs(ec_logits_th, ec_ids, ec_mask, e_batch['chosen_starts'])
                er_log_th = get_sequence_log_probs(er_logits_th, er_ids, er_mask, e_batch['rejected_starts'])

                # Ref forward
                with policy_model.disable_adapter():
                    ec_logits_ref = policy_model(ec_ids, attention_mask=ec_mask).logits
                    er_logits_ref = policy_model(er_ids, attention_mask=er_mask).logits
                    ec_log_ref = get_sequence_log_probs(ec_logits_ref, ec_ids, ec_mask, e_batch['chosen_starts'])
                    er_log_ref = get_sequence_log_probs(er_logits_ref, er_ids, er_mask, e_batch['rejected_starts'])

                c_diff = ec_log_th - ec_log_ref
                r_diff = er_log_th - er_log_ref

                # Metrics
                eval_accs.extend((c_diff > r_diff).float().cpu().numpy())
                # KL is roughly the log ratio between theta and ref. Cast to float32 first!
                eval_kls.extend(c_diff.to(torch.float32).cpu().numpy())

        mean_eval_acc = np.mean(eval_accs)
        mean_eval_kl = np.mean(eval_kls)
        print(f"EVAL | Acc: {mean_eval_acc*100:.1f}% | KL: {mean_eval_kl:.4f}")
        print(f"----------------------------------\n")

    step += 1

print("\nSaving DPO Checkpoint...")
policy_model.save_pretrained("./dpo_checkpoint")
print("DPO Run Complete.")

In [ ]:
import re
import matplotlib.pyplot as plt

# Paste your entire Colab console output between the triple quotes
raw_logs = """
Starting DPO Training (1 Epoch, ~5318 steps)...
Step 0000 | Loss: 0.3008 | Margin: 1.2812 | Acc: 100.0% | C-Len: 56.8 | R-Len: 26.0
Step 0010 | Loss: 1.1094 | Margin: -0.5195 | Acc: 37.5% | C-Len: 42.5 | R-Len: 66.6
Step 0020 | Loss: 0.9922 | Margin: -0.2480 | Acc: 12.5% | C-Len: 32.8 | R-Len: 69.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.0% | KL: 8.0154
----------------------------------

Step 0030 | Loss: 1.0234 | Margin: -0.3359 | Acc: 62.5% | C-Len: 39.1 | R-Len: 33.0
Step 0040 | Loss: 1.0625 | Margin: -0.4648 | Acc: 25.0% | C-Len: 20.8 | R-Len: 39.4
Step 0050 | Loss: 0.8438 | Margin: 0.0645 | Acc: 62.5% | C-Len: 32.0 | R-Len: 64.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.0% | KL: 8.0222
----------------------------------

Step 0060 | Loss: 0.8008 | Margin: 0.0112 | Acc: 50.0% | C-Len: 32.0 | R-Len: 45.1
Step 0070 | Loss: 0.8672 | Margin: -0.0349 | Acc: 62.5% | C-Len: 55.8 | R-Len: 61.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.0% | KL: 8.0330
----------------------------------

Step 0080 | Loss: 0.7617 | Margin: -0.0032 | Acc: 37.5% | C-Len: 27.1 | R-Len: 41.1
Step 0090 | Loss: 0.6445 | Margin: 0.3125 | Acc: 62.5% | C-Len: 27.8 | R-Len: 31.8
Step 0100 | Loss: 0.7539 | Margin: -0.0048 | Acc: 62.5% | C-Len: 45.6 | R-Len: 38.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.0% | KL: 8.0436
----------------------------------

Step 0110 | Loss: 1.1094 | Margin: -0.6523 | Acc: 12.5% | C-Len: 56.4 | R-Len: 59.5
Step 0120 | Loss: 0.6797 | Margin: 0.1865 | Acc: 50.0% | C-Len: 43.1 | R-Len: 56.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.5% | KL: 8.0299
----------------------------------

Step 0130 | Loss: 0.8789 | Margin: -0.2354 | Acc: 25.0% | C-Len: 27.9 | R-Len: 42.1
Step 0140 | Loss: 0.8594 | Margin: -0.1670 | Acc: 37.5% | C-Len: 33.5 | R-Len: 82.6
Step 0150 | Loss: 1.0781 | Margin: -0.4785 | Acc: 37.5% | C-Len: 30.8 | R-Len: 124.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.5% | KL: 8.0345
----------------------------------

Step 0160 | Loss: 0.9766 | Margin: -0.1221 | Acc: 50.0% | C-Len: 26.6 | R-Len: 65.2
Step 0170 | Loss: 0.5586 | Margin: 0.4805 | Acc: 75.0% | C-Len: 30.8 | R-Len: 19.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.5% | KL: 8.0555
----------------------------------

Step 0180 | Loss: 0.8516 | Margin: -0.2617 | Acc: 25.0% | C-Len: 26.5 | R-Len: 43.5
Step 0190 | Loss: 0.9492 | Margin: -0.3672 | Acc: 37.5% | C-Len: 17.5 | R-Len: 37.6
Step 0200 | Loss: 0.4590 | Margin: 0.5586 | Acc: 100.0% | C-Len: 74.0 | R-Len: 46.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.5% | KL: 8.0148
----------------------------------

Step 0210 | Loss: 0.8320 | Margin: -0.1187 | Acc: 62.5% | C-Len: 24.8 | R-Len: 47.2
Step 0220 | Loss: 0.8086 | Margin: 0.0398 | Acc: 62.5% | C-Len: 39.6 | R-Len: 52.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.0% | KL: 8.0077
----------------------------------

Step 0230 | Loss: 0.6641 | Margin: 0.1875 | Acc: 37.5% | C-Len: 17.8 | R-Len: 29.6
Step 0240 | Loss: 1.1172 | Margin: -0.4258 | Acc: 50.0% | C-Len: 34.9 | R-Len: 53.9
Step 0250 | Loss: 1.0391 | Margin: -0.5156 | Acc: 25.0% | C-Len: 54.6 | R-Len: 74.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.5% | KL: 7.9917
----------------------------------

Step 0260 | Loss: 0.9688 | Margin: -0.3984 | Acc: 37.5% | C-Len: 17.9 | R-Len: 80.1
Step 0270 | Loss: 0.5977 | Margin: 0.3262 | Acc: 62.5% | C-Len: 69.1 | R-Len: 58.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.0% | KL: 7.9794
----------------------------------

Step 0280 | Loss: 0.5117 | Margin: 0.5234 | Acc: 75.0% | C-Len: 54.8 | R-Len: 60.5
Step 0290 | Loss: 0.9688 | Margin: -0.3828 | Acc: 37.5% | C-Len: 29.5 | R-Len: 40.8
Step 0300 | Loss: 0.5039 | Margin: 0.7539 | Acc: 87.5% | C-Len: 67.4 | R-Len: 34.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.5% | KL: 8.0040
----------------------------------

Step 0310 | Loss: 0.7188 | Margin: 0.2217 | Acc: 62.5% | C-Len: 46.9 | R-Len: 61.8
Step 0320 | Loss: 0.8242 | Margin: -0.1934 | Acc: 25.0% | C-Len: 39.0 | R-Len: 66.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.0% | KL: 7.9725
----------------------------------

Step 0330 | Loss: 0.6914 | Margin: 0.1553 | Acc: 50.0% | C-Len: 31.4 | R-Len: 38.4
Step 0340 | Loss: 1.2500 | Margin: -0.5078 | Acc: 50.0% | C-Len: 50.4 | R-Len: 70.5
Step 0350 | Loss: 0.7422 | Margin: 0.0081 | Acc: 50.0% | C-Len: 42.4 | R-Len: 55.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.0% | KL: 7.9839
----------------------------------

Step 0360 | Loss: 0.8320 | Margin: -0.1338 | Acc: 50.0% | C-Len: 23.0 | R-Len: 30.9
Step 0370 | Loss: 0.9648 | Margin: -0.2656 | Acc: 25.0% | C-Len: 31.2 | R-Len: 46.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.0% | KL: 7.9953
----------------------------------

Step 0380 | Loss: 1.0078 | Margin: -0.3984 | Acc: 25.0% | C-Len: 37.2 | R-Len: 63.4
Step 0390 | Loss: 0.9961 | Margin: -0.3867 | Acc: 37.5% | C-Len: 22.0 | R-Len: 49.6
Step 0400 | Loss: 0.7930 | Margin: -0.0012 | Acc: 50.0% | C-Len: 41.5 | R-Len: 52.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 47.0% | KL: 7.9831
----------------------------------

Step 0410 | Loss: 1.2422 | Margin: -0.6719 | Acc: 37.5% | C-Len: 34.0 | R-Len: 65.0
Step 0420 | Loss: 0.7617 | Margin: -0.0206 | Acc: 50.0% | C-Len: 57.4 | R-Len: 46.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.0% | KL: 8.0095
----------------------------------

Step 0430 | Loss: 0.8789 | Margin: -0.1992 | Acc: 75.0% | C-Len: 38.5 | R-Len: 76.0
Step 0440 | Loss: 0.9766 | Margin: -0.3945 | Acc: 25.0% | C-Len: 26.0 | R-Len: 46.1
Step 0450 | Loss: 1.1172 | Margin: -0.5820 | Acc: 37.5% | C-Len: 27.6 | R-Len: 49.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.0% | KL: 7.9340
----------------------------------

Step 0460 | Loss: 0.9844 | Margin: -0.3281 | Acc: 25.0% | C-Len: 34.9 | R-Len: 53.9
Step 0470 | Loss: 0.6289 | Margin: 0.7070 | Acc: 62.5% | C-Len: 61.1 | R-Len: 39.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.5% | KL: 7.9606
----------------------------------

Step 0480 | Loss: 1.0078 | Margin: -0.3477 | Acc: 25.0% | C-Len: 32.4 | R-Len: 44.2
Step 0490 | Loss: 0.7773 | Margin: -0.0918 | Acc: 37.5% | C-Len: 21.2 | R-Len: 29.0
Step 0500 | Loss: 0.8711 | Margin: -0.2930 | Acc: 37.5% | C-Len: 28.8 | R-Len: 62.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.5% | KL: 7.9370
----------------------------------

Step 0510 | Loss: 0.9688 | Margin: -0.4043 | Acc: 25.0% | C-Len: 30.5 | R-Len: 51.2
Step 0520 | Loss: 0.9570 | Margin: -0.4043 | Acc: 25.0% | C-Len: 56.6 | R-Len: 54.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.0% | KL: 7.9520
----------------------------------

Step 0530 | Loss: 0.5352 | Margin: 0.4590 | Acc: 87.5% | C-Len: 49.8 | R-Len: 37.2
Step 0540 | Loss: 1.0312 | Margin: -0.3203 | Acc: 37.5% | C-Len: 23.1 | R-Len: 47.4
Step 0550 | Loss: 0.5820 | Margin: 0.5195 | Acc: 75.0% | C-Len: 43.1 | R-Len: 46.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.5% | KL: 7.8922
----------------------------------

Step 0560 | Loss: 0.6328 | Margin: 0.3691 | Acc: 62.5% | C-Len: 47.2 | R-Len: 37.1
Step 0570 | Loss: 0.5391 | Margin: 0.4473 | Acc: 75.0% | C-Len: 57.2 | R-Len: 32.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 44.5% | KL: 7.9123
----------------------------------

Step 0580 | Loss: 0.8047 | Margin: -0.1562 | Acc: 37.5% | C-Len: 22.0 | R-Len: 40.6
Step 0590 | Loss: 1.0391 | Margin: -0.4375 | Acc: 25.0% | C-Len: 43.4 | R-Len: 75.4
Step 0600 | Loss: 0.8672 | Margin: -0.2891 | Acc: 37.5% | C-Len: 14.8 | R-Len: 43.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.0% | KL: 7.9102
----------------------------------

Step 0610 | Loss: 0.7031 | Margin: 0.0427 | Acc: 62.5% | C-Len: 20.1 | R-Len: 40.2
Step 0620 | Loss: 0.6875 | Margin: 0.1797 | Acc: 37.5% | C-Len: 33.6 | R-Len: 24.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 44.5% | KL: 7.9078
----------------------------------

Step 0630 | Loss: 0.9531 | Margin: -0.4180 | Acc: 25.0% | C-Len: 19.8 | R-Len: 54.0
Step 0640 | Loss: 0.8984 | Margin: -0.3223 | Acc: 25.0% | C-Len: 26.8 | R-Len: 59.9
Step 0650 | Loss: 0.7461 | Margin: 0.0115 | Acc: 50.0% | C-Len: 45.6 | R-Len: 28.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 47.0% | KL: 7.9071
----------------------------------

Step 0660 | Loss: 1.1562 | Margin: -0.4258 | Acc: 25.0% | C-Len: 55.5 | R-Len: 68.5
Step 0670 | Loss: 0.8555 | Margin: -0.1387 | Acc: 37.5% | C-Len: 32.6 | R-Len: 51.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 44.5% | KL: 7.8727
----------------------------------

Step 0680 | Loss: 1.1562 | Margin: -0.4668 | Acc: 37.5% | C-Len: 80.8 | R-Len: 114.0
Step 0690 | Loss: 1.3281 | Margin: -0.6133 | Acc: 37.5% | C-Len: 40.8 | R-Len: 78.1
Step 0700 | Loss: 0.8438 | Margin: -0.1641 | Acc: 50.0% | C-Len: 36.6 | R-Len: 77.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.0% | KL: 7.8741
----------------------------------

Step 0710 | Loss: 0.7227 | Margin: 0.0417 | Acc: 62.5% | C-Len: 47.0 | R-Len: 42.9
Step 0720 | Loss: 0.8984 | Margin: -0.3242 | Acc: 37.5% | C-Len: 31.5 | R-Len: 42.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.5% | KL: 7.8735
----------------------------------

Step 0730 | Loss: 0.7578 | Margin: -0.0613 | Acc: 37.5% | C-Len: 34.4 | R-Len: 51.5
Step 0740 | Loss: 1.2422 | Margin: -0.7617 | Acc: 25.0% | C-Len: 32.2 | R-Len: 64.8
Step 0750 | Loss: 0.9375 | Margin: -0.2305 | Acc: 37.5% | C-Len: 33.9 | R-Len: 74.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 44.5% | KL: 7.8622
----------------------------------

Step 0760 | Loss: 0.4609 | Margin: 0.7383 | Acc: 87.5% | C-Len: 35.8 | R-Len: 33.2
Step 0770 | Loss: 0.8984 | Margin: -0.1758 | Acc: 37.5% | C-Len: 24.4 | R-Len: 29.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.5% | KL: 7.8453
----------------------------------

Step 0780 | Loss: 0.9141 | Margin: -0.2793 | Acc: 50.0% | C-Len: 37.6 | R-Len: 59.2
Step 0790 | Loss: 0.8594 | Margin: -0.2490 | Acc: 50.0% | C-Len: 22.6 | R-Len: 32.8
Step 0800 | Loss: 0.5742 | Margin: 0.3887 | Acc: 75.0% | C-Len: 35.4 | R-Len: 22.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.0% | KL: 7.8213
----------------------------------

Step 0810 | Loss: 1.0938 | Margin: -0.5234 | Acc: 25.0% | C-Len: 26.4 | R-Len: 47.0
Step 0820 | Loss: 0.9102 | Margin: -0.2031 | Acc: 50.0% | C-Len: 48.8 | R-Len: 38.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.5% | KL: 7.8018
----------------------------------

Step 0830 | Loss: 0.8008 | Margin: -0.0889 | Acc: 50.0% | C-Len: 25.2 | R-Len: 28.9
Step 0840 | Loss: 0.9219 | Margin: -0.0718 | Acc: 50.0% | C-Len: 43.0 | R-Len: 41.2
Step 0850 | Loss: 0.7656 | Margin: 0.2891 | Acc: 62.5% | C-Len: 58.4 | R-Len: 55.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.0% | KL: 7.8106
----------------------------------

Step 0860 | Loss: 1.3438 | Margin: -0.8281 | Acc: 37.5% | C-Len: 54.4 | R-Len: 89.0
Step 0870 | Loss: 0.9609 | Margin: -0.3438 | Acc: 37.5% | C-Len: 44.0 | R-Len: 58.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 47.0% | KL: 7.7900
----------------------------------

Step 0880 | Loss: 0.7109 | Margin: 0.1196 | Acc: 37.5% | C-Len: 45.2 | R-Len: 63.8
Step 0890 | Loss: 0.8594 | Margin: -0.0549 | Acc: 62.5% | C-Len: 39.9 | R-Len: 56.1
Step 0900 | Loss: 0.9297 | Margin: -0.1904 | Acc: 37.5% | C-Len: 32.4 | R-Len: 57.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.5% | KL: 7.7541
----------------------------------

Step 0910 | Loss: 0.6836 | Margin: 0.7539 | Acc: 37.5% | C-Len: 65.4 | R-Len: 46.1
Step 0920 | Loss: 0.8672 | Margin: -0.2734 | Acc: 37.5% | C-Len: 24.0 | R-Len: 34.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 44.5% | KL: 7.7097
----------------------------------

Step 0930 | Loss: 0.8203 | Margin: -0.0859 | Acc: 50.0% | C-Len: 45.5 | R-Len: 48.2
Step 0940 | Loss: 0.5859 | Margin: 0.5117 | Acc: 75.0% | C-Len: 44.2 | R-Len: 42.4
Step 0950 | Loss: 0.7617 | Margin: -0.0315 | Acc: 50.0% | C-Len: 30.4 | R-Len: 29.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 44.5% | KL: 7.7451
----------------------------------

Step 0960 | Loss: 0.6641 | Margin: 0.2070 | Acc: 50.0% | C-Len: 40.2 | R-Len: 55.5
Step 0970 | Loss: 0.7109 | Margin: 0.0879 | Acc: 50.0% | C-Len: 32.6 | R-Len: 32.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.5% | KL: 7.7206
----------------------------------

Step 0980 | Loss: 0.7188 | Margin: 0.0845 | Acc: 37.5% | C-Len: 28.8 | R-Len: 28.1
Step 0990 | Loss: 0.8398 | Margin: -0.1157 | Acc: 75.0% | C-Len: 30.4 | R-Len: 56.8
Step 1000 | Loss: 0.7969 | Margin: 0.1445 | Acc: 50.0% | C-Len: 56.0 | R-Len: 58.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 45.0% | KL: 7.7166
----------------------------------

Step 1010 | Loss: 0.6523 | Margin: 0.1133 | Acc: 75.0% | C-Len: 30.9 | R-Len: 38.8
Step 1020 | Loss: 0.7930 | Margin: -0.1133 | Acc: 50.0% | C-Len: 43.6 | R-Len: 55.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.5% | KL: 7.6737
----------------------------------

Step 1030 | Loss: 0.9531 | Margin: -0.1855 | Acc: 25.0% | C-Len: 26.5 | R-Len: 37.8
Step 1040 | Loss: 0.7812 | Margin: 0.0537 | Acc: 37.5% | C-Len: 39.9 | R-Len: 58.9
Step 1050 | Loss: 0.7031 | Margin: 0.1670 | Acc: 62.5% | C-Len: 30.2 | R-Len: 38.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 47.0% | KL: 7.6917
----------------------------------

Step 1060 | Loss: 0.6914 | Margin: 0.1514 | Acc: 62.5% | C-Len: 23.9 | R-Len: 73.4
Step 1070 | Loss: 0.7266 | Margin: 0.0260 | Acc: 75.0% | C-Len: 20.5 | R-Len: 28.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.5% | KL: 7.6176
----------------------------------

Step 1080 | Loss: 0.9492 | Margin: -0.2715 | Acc: 37.5% | C-Len: 35.2 | R-Len: 33.5
Step 1090 | Loss: 1.1953 | Margin: -0.7578 | Acc: 12.5% | C-Len: 14.1 | R-Len: 24.6
Step 1100 | Loss: 0.9414 | Margin: -0.0981 | Acc: 25.0% | C-Len: 25.9 | R-Len: 49.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.0% | KL: 7.6578
----------------------------------

Step 1110 | Loss: 0.6523 | Margin: 0.1553 | Acc: 62.5% | C-Len: 22.9 | R-Len: 28.5
Step 1120 | Loss: 0.7227 | Margin: 0.0176 | Acc: 50.0% | C-Len: 26.4 | R-Len: 65.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 47.5% | KL: 7.6110
----------------------------------

Step 1130 | Loss: 0.8398 | Margin: -0.1543 | Acc: 62.5% | C-Len: 30.0 | R-Len: 53.1
Step 1140 | Loss: 0.7539 | Margin: 0.1982 | Acc: 50.0% | C-Len: 76.6 | R-Len: 65.9
Step 1150 | Loss: 0.7227 | Margin: 0.1836 | Acc: 37.5% | C-Len: 52.5 | R-Len: 36.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 47.0% | KL: 7.5690
----------------------------------

Step 1160 | Loss: 0.7695 | Margin: 0.0981 | Acc: 62.5% | C-Len: 32.8 | R-Len: 32.8
Step 1170 | Loss: 0.6641 | Margin: 0.1934 | Acc: 62.5% | C-Len: 25.4 | R-Len: 19.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.5% | KL: 7.5679
----------------------------------

Step 1180 | Loss: 0.6406 | Margin: 0.1768 | Acc: 62.5% | C-Len: 54.9 | R-Len: 49.9
Step 1190 | Loss: 0.7539 | Margin: -0.0583 | Acc: 25.0% | C-Len: 37.4 | R-Len: 45.0
Step 1200 | Loss: 0.7617 | Margin: -0.0679 | Acc: 50.0% | C-Len: 30.0 | R-Len: 55.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 46.5% | KL: 7.5148
----------------------------------

Step 1210 | Loss: 0.9336 | Margin: -0.3828 | Acc: 12.5% | C-Len: 29.2 | R-Len: 70.5
Step 1220 | Loss: 0.7969 | Margin: 0.1040 | Acc: 37.5% | C-Len: 34.5 | R-Len: 56.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 47.5% | KL: 7.5272
----------------------------------

Step 1230 | Loss: 0.7812 | Margin: -0.0371 | Acc: 37.5% | C-Len: 25.4 | R-Len: 51.6
Step 1240 | Loss: 0.6953 | Margin: 0.1113 | Acc: 37.5% | C-Len: 25.1 | R-Len: 47.2
Step 1250 | Loss: 0.6055 | Margin: 0.4414 | Acc: 87.5% | C-Len: 34.6 | R-Len: 50.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.0% | KL: 7.4601
----------------------------------

Step 1260 | Loss: 0.6719 | Margin: 0.0879 | Acc: 50.0% | C-Len: 39.9 | R-Len: 51.8
Step 1270 | Loss: 0.7812 | Margin: -0.0554 | Acc: 62.5% | C-Len: 18.9 | R-Len: 37.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.0% | KL: 7.4425
----------------------------------

Step 1280 | Loss: 1.0156 | Margin: -0.1895 | Acc: 25.0% | C-Len: 74.8 | R-Len: 97.4
Step 1290 | Loss: 0.6172 | Margin: 0.3164 | Acc: 62.5% | C-Len: 46.8 | R-Len: 58.6
Step 1300 | Loss: 0.6172 | Margin: 0.2539 | Acc: 62.5% | C-Len: 44.5 | R-Len: 34.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.0% | KL: 7.4426
----------------------------------

Step 1310 | Loss: 0.6875 | Margin: 0.1240 | Acc: 50.0% | C-Len: 30.0 | R-Len: 35.8
Step 1320 | Loss: 0.7344 | Margin: 0.0437 | Acc: 62.5% | C-Len: 39.1 | R-Len: 69.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.0% | KL: 7.3902
----------------------------------

Step 1330 | Loss: 0.7578 | Margin: -0.0322 | Acc: 25.0% | C-Len: 42.2 | R-Len: 72.5
Step 1340 | Loss: 0.8398 | Margin: -0.1602 | Acc: 37.5% | C-Len: 41.9 | R-Len: 60.5
Step 1350 | Loss: 0.6797 | Margin: 0.1187 | Acc: 50.0% | C-Len: 85.1 | R-Len: 75.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.0% | KL: 7.3511
----------------------------------

Step 1360 | Loss: 0.7969 | Margin: 0.1895 | Acc: 62.5% | C-Len: 26.2 | R-Len: 45.4
Step 1370 | Loss: 0.7344 | Margin: 0.0308 | Acc: 50.0% | C-Len: 34.8 | R-Len: 54.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 47.5% | KL: 7.3516
----------------------------------

Step 1380 | Loss: 0.6875 | Margin: 0.1650 | Acc: 62.5% | C-Len: 58.1 | R-Len: 50.9
Step 1390 | Loss: 1.0234 | Margin: -0.5000 | Acc: 37.5% | C-Len: 29.0 | R-Len: 37.0
Step 1400 | Loss: 0.8711 | Margin: -0.0625 | Acc: 62.5% | C-Len: 60.4 | R-Len: 52.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.0% | KL: 7.3230
----------------------------------

Step 1410 | Loss: 0.6445 | Margin: 0.3770 | Acc: 50.0% | C-Len: 54.5 | R-Len: 68.0
Step 1420 | Loss: 0.7578 | Margin: -0.0596 | Acc: 62.5% | C-Len: 42.1 | R-Len: 57.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.5% | KL: 7.3155
----------------------------------

Step 1430 | Loss: 0.6133 | Margin: 0.3047 | Acc: 50.0% | C-Len: 48.5 | R-Len: 50.8
Step 1440 | Loss: 0.6562 | Margin: 0.3008 | Acc: 50.0% | C-Len: 40.8 | R-Len: 41.2
Step 1450 | Loss: 0.7422 | Margin: 0.0981 | Acc: 50.0% | C-Len: 39.2 | R-Len: 56.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 47.5% | KL: 7.2681
----------------------------------

Step 1460 | Loss: 0.6953 | Margin: 0.0615 | Acc: 50.0% | C-Len: 54.9 | R-Len: 66.1
Step 1470 | Loss: 0.8633 | Margin: 0.0181 | Acc: 37.5% | C-Len: 52.6 | R-Len: 40.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.5% | KL: 7.2056
----------------------------------

Step 1480 | Loss: 0.9180 | Margin: -0.3320 | Acc: 37.5% | C-Len: 24.0 | R-Len: 44.4
Step 1490 | Loss: 0.5898 | Margin: 0.3242 | Acc: 87.5% | C-Len: 51.8 | R-Len: 67.9
Step 1500 | Loss: 0.7617 | Margin: -0.0894 | Acc: 25.0% | C-Len: 33.5 | R-Len: 37.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.0% | KL: 7.1739
----------------------------------

Step 1510 | Loss: 0.8711 | Margin: -0.2969 | Acc: 12.5% | C-Len: 31.2 | R-Len: 74.4
Step 1520 | Loss: 0.6680 | Margin: 0.3438 | Acc: 75.0% | C-Len: 36.5 | R-Len: 43.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.0% | KL: 7.1444
----------------------------------

Step 1530 | Loss: 0.7891 | Margin: 0.0566 | Acc: 37.5% | C-Len: 47.4 | R-Len: 60.9
Step 1540 | Loss: 0.6953 | Margin: 0.2275 | Acc: 62.5% | C-Len: 55.9 | R-Len: 52.2
Step 1550 | Loss: 0.6328 | Margin: 0.2412 | Acc: 62.5% | C-Len: 44.8 | R-Len: 58.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.5% | KL: 7.1046
----------------------------------

Step 1560 | Loss: 0.5938 | Margin: 0.2754 | Acc: 75.0% | C-Len: 26.5 | R-Len: 20.4
Step 1570 | Loss: 0.7891 | Margin: 0.1475 | Acc: 62.5% | C-Len: 42.8 | R-Len: 55.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 47.5% | KL: 7.0848
----------------------------------

Step 1580 | Loss: 0.6094 | Margin: 0.3301 | Acc: 75.0% | C-Len: 38.8 | R-Len: 42.6
Step 1590 | Loss: 0.6367 | Margin: 0.1826 | Acc: 62.5% | C-Len: 81.1 | R-Len: 54.4
Step 1600 | Loss: 0.6875 | Margin: 0.1309 | Acc: 50.0% | C-Len: 34.2 | R-Len: 24.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.5% | KL: 7.0350
----------------------------------

Step 1610 | Loss: 0.7969 | Margin: -0.0776 | Acc: 37.5% | C-Len: 47.1 | R-Len: 34.8
Step 1620 | Loss: 0.7930 | Margin: -0.0077 | Acc: 37.5% | C-Len: 62.2 | R-Len: 43.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 50.0% | KL: 6.9822
----------------------------------

Step 1630 | Loss: 0.6992 | Margin: 0.0771 | Acc: 62.5% | C-Len: 35.0 | R-Len: 57.9
Step 1640 | Loss: 0.6484 | Margin: 0.1128 | Acc: 62.5% | C-Len: 28.8 | R-Len: 44.1
Step 1650 | Loss: 0.7422 | Margin: -0.0613 | Acc: 62.5% | C-Len: 24.5 | R-Len: 33.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.5% | KL: 6.9400
----------------------------------

Step 1660 | Loss: 0.8633 | Margin: -0.1865 | Acc: 37.5% | C-Len: 30.8 | R-Len: 60.4
Step 1670 | Loss: 1.0547 | Margin: -0.5039 | Acc: 25.0% | C-Len: 40.5 | R-Len: 77.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.0% | KL: 6.8985
----------------------------------

Step 1680 | Loss: 0.7383 | Margin: 0.0454 | Acc: 25.0% | C-Len: 27.9 | R-Len: 53.8
Step 1690 | Loss: 0.7148 | Margin: 0.1133 | Acc: 50.0% | C-Len: 52.9 | R-Len: 33.5
Step 1700 | Loss: 0.9375 | Margin: -0.1885 | Acc: 37.5% | C-Len: 32.0 | R-Len: 59.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.0% | KL: 6.8411
----------------------------------

Step 1710 | Loss: 0.8398 | Margin: -0.2246 | Acc: 37.5% | C-Len: 23.9 | R-Len: 39.5
Step 1720 | Loss: 0.6719 | Margin: 0.0752 | Acc: 37.5% | C-Len: 30.5 | R-Len: 42.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 50.0% | KL: 6.7960
----------------------------------

Step 1730 | Loss: 0.7812 | Margin: 0.2295 | Acc: 37.5% | C-Len: 53.0 | R-Len: 60.2
Step 1740 | Loss: 0.6133 | Margin: 0.4121 | Acc: 37.5% | C-Len: 40.2 | R-Len: 37.0
Step 1750 | Loss: 0.6719 | Margin: 0.3691 | Acc: 75.0% | C-Len: 44.6 | R-Len: 27.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.0% | KL: 6.7587
----------------------------------

Step 1760 | Loss: 0.8086 | Margin: -0.1523 | Acc: 50.0% | C-Len: 30.2 | R-Len: 65.1
Step 1770 | Loss: 0.8711 | Margin: -0.2334 | Acc: 37.5% | C-Len: 29.2 | R-Len: 37.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.5% | KL: 6.7002
----------------------------------

Step 1780 | Loss: 0.6016 | Margin: 0.2871 | Acc: 75.0% | C-Len: 57.5 | R-Len: 35.8
Step 1790 | Loss: 0.6406 | Margin: 0.2910 | Acc: 50.0% | C-Len: 37.0 | R-Len: 30.1
Step 1800 | Loss: 0.9648 | Margin: -0.2949 | Acc: 25.0% | C-Len: 33.9 | R-Len: 39.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 50.0% | KL: 6.6671
----------------------------------

Step 1810 | Loss: 0.7539 | Margin: -0.0050 | Acc: 50.0% | C-Len: 17.9 | R-Len: 21.0
Step 1820 | Loss: 0.6523 | Margin: 0.3887 | Acc: 62.5% | C-Len: 47.8 | R-Len: 53.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 50.5% | KL: 6.6251
----------------------------------

Step 1830 | Loss: 0.5352 | Margin: 0.3965 | Acc: 75.0% | C-Len: 28.2 | R-Len: 38.6
Step 1840 | Loss: 0.8984 | Margin: -0.1895 | Acc: 37.5% | C-Len: 27.8 | R-Len: 50.0
Step 1850 | Loss: 0.8438 | Margin: -0.1670 | Acc: 25.0% | C-Len: 45.2 | R-Len: 67.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.5% | KL: 6.5875
----------------------------------

Step 1860 | Loss: 0.7383 | Margin: 0.0369 | Acc: 50.0% | C-Len: 19.4 | R-Len: 23.6
Step 1870 | Loss: 0.5703 | Margin: 0.3594 | Acc: 75.0% | C-Len: 38.2 | R-Len: 40.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.0% | KL: 6.5030
----------------------------------

Step 1880 | Loss: 0.5586 | Margin: 0.4902 | Acc: 62.5% | C-Len: 41.4 | R-Len: 44.9
Step 1890 | Loss: 0.8047 | Margin: -0.0515 | Acc: 50.0% | C-Len: 33.9 | R-Len: 46.9
Step 1900 | Loss: 0.6445 | Margin: 0.1719 | Acc: 75.0% | C-Len: 24.6 | R-Len: 30.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.5% | KL: 6.4735
----------------------------------

Step 1910 | Loss: 0.5703 | Margin: 0.3711 | Acc: 75.0% | C-Len: 45.0 | R-Len: 60.6
Step 1920 | Loss: 0.9023 | Margin: -0.0713 | Acc: 50.0% | C-Len: 40.0 | R-Len: 36.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.0% | KL: 6.3661
----------------------------------

Step 1930 | Loss: 0.7305 | Margin: 0.0266 | Acc: 37.5% | C-Len: 23.0 | R-Len: 56.6
Step 1940 | Loss: 0.9180 | Margin: -0.3125 | Acc: 37.5% | C-Len: 25.5 | R-Len: 70.1
Step 1950 | Loss: 0.8867 | Margin: -0.3281 | Acc: 25.0% | C-Len: 26.4 | R-Len: 26.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 50.5% | KL: 6.2619
----------------------------------

Step 1960 | Loss: 0.5469 | Margin: 0.3848 | Acc: 87.5% | C-Len: 33.2 | R-Len: 40.0
Step 1970 | Loss: 0.6875 | Margin: 0.2168 | Acc: 50.0% | C-Len: 34.0 | R-Len: 35.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 48.5% | KL: 6.1757
----------------------------------

Step 1980 | Loss: 0.7969 | Margin: -0.0728 | Acc: 50.0% | C-Len: 33.5 | R-Len: 43.5
Step 1990 | Loss: 0.8633 | Margin: 0.0072 | Acc: 50.0% | C-Len: 54.6 | R-Len: 44.1
Step 2000 | Loss: 0.8086 | Margin: 0.0608 | Acc: 50.0% | C-Len: 61.2 | R-Len: 41.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 51.0% | KL: 6.1416
----------------------------------

Step 2010 | Loss: 0.9531 | Margin: -0.2871 | Acc: 37.5% | C-Len: 29.5 | R-Len: 43.2
Step 2020 | Loss: 0.7852 | Margin: 0.0212 | Acc: 62.5% | C-Len: 66.9 | R-Len: 40.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.0% | KL: 6.0849
----------------------------------

Step 2030 | Loss: 0.7227 | Margin: 0.1230 | Acc: 37.5% | C-Len: 33.6 | R-Len: 32.2
Step 2040 | Loss: 0.9336 | Margin: -0.1050 | Acc: 62.5% | C-Len: 47.9 | R-Len: 42.9
Step 2050 | Loss: 0.7734 | Margin: -0.0233 | Acc: 62.5% | C-Len: 26.2 | R-Len: 56.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 50.0% | KL: 6.0044
----------------------------------

Step 2060 | Loss: 0.7461 | Margin: -0.0172 | Acc: 37.5% | C-Len: 17.4 | R-Len: 30.2
Step 2070 | Loss: 0.7891 | Margin: -0.1143 | Acc: 50.0% | C-Len: 25.9 | R-Len: 43.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 51.0% | KL: 5.9860
----------------------------------

Step 2080 | Loss: 0.6836 | Margin: 0.0537 | Acc: 62.5% | C-Len: 13.6 | R-Len: 31.9
Step 2090 | Loss: 0.7852 | Margin: -0.0139 | Acc: 50.0% | C-Len: 28.4 | R-Len: 37.5
Step 2100 | Loss: 0.7109 | Margin: 0.1338 | Acc: 50.0% | C-Len: 55.2 | R-Len: 42.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 50.5% | KL: 5.9009
----------------------------------

Step 2110 | Loss: 0.7305 | Margin: 0.0058 | Acc: 50.0% | C-Len: 44.9 | R-Len: 41.0
Step 2120 | Loss: 0.6172 | Margin: 0.4199 | Acc: 62.5% | C-Len: 26.6 | R-Len: 25.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.5% | KL: 5.8420
----------------------------------

Step 2130 | Loss: 1.1094 | Margin: -0.5430 | Acc: 37.5% | C-Len: 42.9 | R-Len: 68.4
Step 2140 | Loss: 0.6953 | Margin: 0.0698 | Acc: 37.5% | C-Len: 19.8 | R-Len: 36.9
Step 2150 | Loss: 0.9805 | Margin: -0.4258 | Acc: 37.5% | C-Len: 45.5 | R-Len: 29.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.5% | KL: 5.7928
----------------------------------

Step 2160 | Loss: 0.6016 | Margin: 0.6523 | Acc: 75.0% | C-Len: 25.6 | R-Len: 24.0
Step 2170 | Loss: 0.6133 | Margin: 0.4434 | Acc: 50.0% | C-Len: 35.5 | R-Len: 66.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 51.0% | KL: 5.6486
----------------------------------

Step 2180 | Loss: 0.6719 | Margin: 0.1582 | Acc: 62.5% | C-Len: 46.9 | R-Len: 38.4
Step 2190 | Loss: 0.9219 | Margin: -0.1885 | Acc: 25.0% | C-Len: 29.1 | R-Len: 48.6
Step 2200 | Loss: 0.5938 | Margin: 0.4922 | Acc: 75.0% | C-Len: 42.6 | R-Len: 47.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 50.5% | KL: 5.5694
----------------------------------

Step 2210 | Loss: 0.6953 | Margin: 0.0270 | Acc: 37.5% | C-Len: 50.5 | R-Len: 27.2
Step 2220 | Loss: 0.7969 | Margin: -0.0306 | Acc: 50.0% | C-Len: 25.2 | R-Len: 76.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 50.5% | KL: 5.4900
----------------------------------

Step 2230 | Loss: 0.5742 | Margin: 0.3223 | Acc: 87.5% | C-Len: 17.4 | R-Len: 17.6
Step 2240 | Loss: 0.8086 | Margin: -0.1611 | Acc: 37.5% | C-Len: 34.9 | R-Len: 27.1
Step 2250 | Loss: 0.9297 | Margin: -0.4043 | Acc: 0.0% | C-Len: 37.0 | R-Len: 52.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 49.5% | KL: 5.3932
----------------------------------

Step 2260 | Loss: 0.6055 | Margin: 0.3594 | Acc: 62.5% | C-Len: 57.6 | R-Len: 39.1
Step 2270 | Loss: 0.6992 | Margin: 0.0449 | Acc: 62.5% | C-Len: 36.8 | R-Len: 50.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 52.0% | KL: 5.2857
----------------------------------

Step 2280 | Loss: 0.5898 | Margin: 0.3848 | Acc: 75.0% | C-Len: 28.9 | R-Len: 35.8
Step 2290 | Loss: 0.5469 | Margin: 0.5742 | Acc: 75.0% | C-Len: 35.2 | R-Len: 32.9
Step 2300 | Loss: 0.5859 | Margin: 0.3008 | Acc: 75.0% | C-Len: 26.6 | R-Len: 47.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 51.5% | KL: 5.1968
----------------------------------

Step 2310 | Loss: 0.8477 | Margin: -0.0830 | Acc: 50.0% | C-Len: 16.4 | R-Len: 28.9
Step 2320 | Loss: 0.5469 | Margin: 0.3457 | Acc: 87.5% | C-Len: 41.2 | R-Len: 26.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 51.0% | KL: 5.0589
----------------------------------

Step 2330 | Loss: 0.6758 | Margin: 0.1260 | Acc: 75.0% | C-Len: 24.6 | R-Len: 40.8
Step 2340 | Loss: 0.9766 | Margin: -0.3867 | Acc: 12.5% | C-Len: 17.0 | R-Len: 27.4
Step 2350 | Loss: 0.7109 | Margin: 0.1494 | Acc: 37.5% | C-Len: 47.2 | R-Len: 42.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 52.0% | KL: 4.9840
----------------------------------

Step 2360 | Loss: 0.5898 | Margin: 0.4746 | Acc: 62.5% | C-Len: 52.4 | R-Len: 45.4
Step 2370 | Loss: 1.0703 | Margin: -0.5078 | Acc: 25.0% | C-Len: 33.0 | R-Len: 76.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 52.0% | KL: 4.8282
----------------------------------

Step 2380 | Loss: 0.5430 | Margin: 0.4004 | Acc: 87.5% | C-Len: 46.6 | R-Len: 67.9
Step 2390 | Loss: 0.8906 | Margin: -0.2617 | Acc: 37.5% | C-Len: 17.0 | R-Len: 88.5
Step 2400 | Loss: 0.6445 | Margin: 0.2412 | Acc: 62.5% | C-Len: 48.2 | R-Len: 59.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 52.0% | KL: 4.7264
----------------------------------

Step 2410 | Loss: 0.8008 | Margin: -0.0447 | Acc: 37.5% | C-Len: 26.9 | R-Len: 61.1
Step 2420 | Loss: 0.7578 | Margin: -0.0859 | Acc: 50.0% | C-Len: 32.8 | R-Len: 57.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 51.5% | KL: 4.5840
----------------------------------

Step 2430 | Loss: 0.6250 | Margin: 0.4707 | Acc: 62.5% | C-Len: 24.4 | R-Len: 53.1
Step 2440 | Loss: 0.6289 | Margin: 0.1836 | Acc: 50.0% | C-Len: 47.8 | R-Len: 62.2
Step 2450 | Loss: 0.8398 | Margin: 0.0654 | Acc: 62.5% | C-Len: 50.6 | R-Len: 87.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 52.5% | KL: 4.4059
----------------------------------

Step 2460 | Loss: 0.8633 | Margin: -0.1973 | Acc: 50.0% | C-Len: 39.2 | R-Len: 31.0
Step 2470 | Loss: 0.8672 | Margin: -0.1816 | Acc: 50.0% | C-Len: 31.9 | R-Len: 83.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 54.5% | KL: 4.2580
----------------------------------

Step 2480 | Loss: 1.1406 | Margin: -0.5078 | Acc: 50.0% | C-Len: 46.0 | R-Len: 96.5
Step 2490 | Loss: 0.7656 | Margin: -0.0369 | Acc: 50.0% | C-Len: 21.2 | R-Len: 77.6
Step 2500 | Loss: 0.7695 | Margin: -0.0757 | Acc: 50.0% | C-Len: 54.9 | R-Len: 48.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 52.0% | KL: 4.0918
----------------------------------

Step 2510 | Loss: 0.6055 | Margin: 0.2441 | Acc: 62.5% | C-Len: 49.5 | R-Len: 49.6
Step 2520 | Loss: 0.5977 | Margin: 0.2656 | Acc: 75.0% | C-Len: 19.0 | R-Len: 28.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 53.0% | KL: 3.9410
----------------------------------

Step 2530 | Loss: 0.9023 | Margin: -0.3340 | Acc: 12.5% | C-Len: 28.2 | R-Len: 58.1
Step 2540 | Loss: 1.0938 | Margin: -0.4395 | Acc: 50.0% | C-Len: 49.0 | R-Len: 52.6
Step 2550 | Loss: 0.7227 | Margin: -0.0308 | Acc: 50.0% | C-Len: 17.0 | R-Len: 29.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 52.5% | KL: 3.7738
----------------------------------

Step 2560 | Loss: 0.7617 | Margin: 0.0200 | Acc: 50.0% | C-Len: 47.0 | R-Len: 58.9
Step 2570 | Loss: 0.5938 | Margin: 0.3652 | Acc: 62.5% | C-Len: 32.5 | R-Len: 64.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 51.5% | KL: 3.5285
----------------------------------

Step 2580 | Loss: 0.5000 | Margin: 0.6250 | Acc: 75.0% | C-Len: 30.6 | R-Len: 26.1
Step 2590 | Loss: 0.8047 | Margin: -0.0476 | Acc: 50.0% | C-Len: 32.6 | R-Len: 56.0
Step 2600 | Loss: 0.8398 | Margin: 0.0117 | Acc: 37.5% | C-Len: 58.5 | R-Len: 63.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 52.5% | KL: 3.3657
----------------------------------

Step 2610 | Loss: 0.4824 | Margin: 0.5195 | Acc: 87.5% | C-Len: 46.9 | R-Len: 48.9
Step 2620 | Loss: 0.9570 | Margin: -0.3691 | Acc: 37.5% | C-Len: 40.6 | R-Len: 52.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 52.5% | KL: 3.1604
----------------------------------

Step 2630 | Loss: 0.5234 | Margin: 0.5625 | Acc: 75.0% | C-Len: 21.9 | R-Len: 58.0
Step 2640 | Loss: 0.9297 | Margin: -0.3086 | Acc: 50.0% | C-Len: 35.2 | R-Len: 45.0
Step 2650 | Loss: 0.7227 | Margin: 0.1846 | Acc: 50.0% | C-Len: 39.4 | R-Len: 42.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 54.0% | KL: 2.9785
----------------------------------

Step 2660 | Loss: 0.4980 | Margin: 0.5469 | Acc: 75.0% | C-Len: 35.5 | R-Len: 51.1
Step 2670 | Loss: 0.8516 | Margin: 0.0154 | Acc: 62.5% | C-Len: 20.4 | R-Len: 56.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 54.0% | KL: 2.7070
----------------------------------

Step 2680 | Loss: 0.9336 | Margin: -0.3281 | Acc: 37.5% | C-Len: 36.4 | R-Len: 62.9
Step 2690 | Loss: 0.7070 | Margin: 0.0742 | Acc: 50.0% | C-Len: 25.8 | R-Len: 41.8
Step 2700 | Loss: 0.8750 | Margin: -0.2871 | Acc: 25.0% | C-Len: 22.1 | R-Len: 33.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 53.5% | KL: 2.4844
----------------------------------

Step 2710 | Loss: 0.6289 | Margin: 0.2461 | Acc: 62.5% | C-Len: 36.4 | R-Len: 28.6
Step 2720 | Loss: 0.6797 | Margin: 0.2051 | Acc: 75.0% | C-Len: 50.0 | R-Len: 37.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 53.5% | KL: 2.3210
----------------------------------

Step 2730 | Loss: 0.6055 | Margin: 0.2812 | Acc: 62.5% | C-Len: 30.9 | R-Len: 52.2
Step 2740 | Loss: 0.9180 | Margin: -0.2236 | Acc: 37.5% | C-Len: 16.0 | R-Len: 47.6
Step 2750 | Loss: 0.6328 | Margin: 0.2969 | Acc: 62.5% | C-Len: 45.1 | R-Len: 36.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 55.0% | KL: 2.1118
----------------------------------

Step 2760 | Loss: 0.6367 | Margin: 0.2041 | Acc: 50.0% | C-Len: 52.5 | R-Len: 35.4
Step 2770 | Loss: 0.8125 | Margin: -0.0542 | Acc: 62.5% | C-Len: 28.1 | R-Len: 60.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 54.0% | KL: 1.9068
----------------------------------

Step 2780 | Loss: 0.6016 | Margin: 0.3105 | Acc: 75.0% | C-Len: 52.6 | R-Len: 64.9
Step 2790 | Loss: 0.5859 | Margin: 0.4102 | Acc: 75.0% | C-Len: 44.5 | R-Len: 66.0
Step 2800 | Loss: 0.4766 | Margin: 0.6914 | Acc: 87.5% | C-Len: 34.2 | R-Len: 50.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 55.5% | KL: 1.7253
----------------------------------

Step 2810 | Loss: 0.6797 | Margin: 0.1445 | Acc: 50.0% | C-Len: 23.9 | R-Len: 71.9
Step 2820 | Loss: 0.9258 | Margin: -0.2637 | Acc: 37.5% | C-Len: 45.0 | R-Len: 33.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 55.0% | KL: 1.4981
----------------------------------

Step 2830 | Loss: 0.7891 | Margin: -0.0728 | Acc: 50.0% | C-Len: 50.6 | R-Len: 36.9
Step 2840 | Loss: 0.5625 | Margin: 0.3867 | Acc: 62.5% | C-Len: 30.2 | R-Len: 60.5
Step 2850 | Loss: 0.5742 | Margin: 0.6094 | Acc: 50.0% | C-Len: 31.9 | R-Len: 51.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 55.5% | KL: 1.2506
----------------------------------

Step 2860 | Loss: 0.9922 | Margin: -0.2793 | Acc: 37.5% | C-Len: 37.2 | R-Len: 87.1
Step 2870 | Loss: 0.7617 | Margin: 0.0156 | Acc: 37.5% | C-Len: 46.6 | R-Len: 52.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 56.0% | KL: 1.0453
----------------------------------

Step 2880 | Loss: 0.6797 | Margin: 0.2383 | Acc: 50.0% | C-Len: 51.2 | R-Len: 62.5
Step 2890 | Loss: 0.6250 | Margin: 0.5078 | Acc: 62.5% | C-Len: 44.4 | R-Len: 62.2
Step 2900 | Loss: 0.7109 | Margin: 0.4629 | Acc: 50.0% | C-Len: 28.0 | R-Len: 93.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 56.5% | KL: 0.8619
----------------------------------

Step 2910 | Loss: 0.8125 | Margin: 0.1030 | Acc: 62.5% | C-Len: 29.9 | R-Len: 63.0
Step 2920 | Loss: 0.5898 | Margin: 0.3047 | Acc: 62.5% | C-Len: 28.6 | R-Len: 88.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.0% | KL: 0.6789
----------------------------------

Step 2930 | Loss: 0.6953 | Margin: 0.1494 | Acc: 37.5% | C-Len: 29.6 | R-Len: 39.8
Step 2940 | Loss: 0.5195 | Margin: 0.6562 | Acc: 75.0% | C-Len: 15.4 | R-Len: 27.6
Step 2950 | Loss: 0.8984 | Margin: -0.1494 | Acc: 50.0% | C-Len: 39.1 | R-Len: 91.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 56.0% | KL: 0.4469
----------------------------------

Step 2960 | Loss: 0.8828 | Margin: -0.0391 | Acc: 37.5% | C-Len: 55.1 | R-Len: 75.9
Step 2970 | Loss: 0.6172 | Margin: 0.2969 | Acc: 62.5% | C-Len: 20.9 | R-Len: 46.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.0% | KL: 0.3050
----------------------------------

Step 2980 | Loss: 0.6367 | Margin: 0.6289 | Acc: 75.0% | C-Len: 28.1 | R-Len: 47.1
Step 2990 | Loss: 0.5195 | Margin: 0.5078 | Acc: 62.5% | C-Len: 29.6 | R-Len: 32.0
Step 3000 | Loss: 0.5234 | Margin: 0.7773 | Acc: 75.0% | C-Len: 14.6 | R-Len: 52.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 55.0% | KL: 0.1379
----------------------------------

Step 3010 | Loss: 0.7773 | Margin: -0.1099 | Acc: 25.0% | C-Len: 31.9 | R-Len: 69.9
Step 3020 | Loss: 0.5820 | Margin: 0.5039 | Acc: 62.5% | C-Len: 37.2 | R-Len: 33.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: 0.0489
----------------------------------

Step 3030 | Loss: 0.4570 | Margin: 0.9531 | Acc: 75.0% | C-Len: 28.6 | R-Len: 26.8
Step 3040 | Loss: 0.6367 | Margin: 0.2832 | Acc: 50.0% | C-Len: 20.8 | R-Len: 33.9
Step 3050 | Loss: 0.8945 | Margin: -0.0471 | Acc: 62.5% | C-Len: 23.5 | R-Len: 66.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 56.5% | KL: -0.0359
----------------------------------

Step 3060 | Loss: 0.7500 | Margin: -0.0054 | Acc: 25.0% | C-Len: 34.2 | R-Len: 31.6
Step 3070 | Loss: 0.6094 | Margin: 0.6055 | Acc: 62.5% | C-Len: 50.8 | R-Len: 27.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -0.1785
----------------------------------

Step 3080 | Loss: 0.8477 | Margin: -0.1406 | Acc: 50.0% | C-Len: 23.2 | R-Len: 44.2
Step 3090 | Loss: 0.7695 | Margin: -0.0294 | Acc: 50.0% | C-Len: 39.2 | R-Len: 44.8
Step 3100 | Loss: 0.6172 | Margin: 0.3418 | Acc: 62.5% | C-Len: 32.8 | R-Len: 56.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -0.2250
----------------------------------

Step 3110 | Loss: 0.3633 | Margin: 0.8750 | Acc: 100.0% | C-Len: 38.2 | R-Len: 49.0
Step 3120 | Loss: 0.5156 | Margin: 0.5977 | Acc: 87.5% | C-Len: 26.0 | R-Len: 46.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -0.3402
----------------------------------

Step 3130 | Loss: 0.5586 | Margin: 0.3711 | Acc: 87.5% | C-Len: 26.8 | R-Len: 52.4
Step 3140 | Loss: 0.6289 | Margin: 0.2207 | Acc: 50.0% | C-Len: 24.6 | R-Len: 32.6
Step 3150 | Loss: 0.6953 | Margin: 0.1982 | Acc: 62.5% | C-Len: 26.8 | R-Len: 62.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.0% | KL: -0.3963
----------------------------------

Step 3160 | Loss: 0.4688 | Margin: 0.6953 | Acc: 87.5% | C-Len: 34.1 | R-Len: 57.1
Step 3170 | Loss: 0.7812 | Margin: -0.0752 | Acc: 62.5% | C-Len: 37.2 | R-Len: 39.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -0.4948
----------------------------------

Step 3180 | Loss: 0.6836 | Margin: 0.1973 | Acc: 50.0% | C-Len: 34.6 | R-Len: 46.9
Step 3190 | Loss: 0.6367 | Margin: 0.4023 | Acc: 62.5% | C-Len: 35.0 | R-Len: 37.1
Step 3200 | Loss: 0.4961 | Margin: 0.6562 | Acc: 75.0% | C-Len: 56.4 | R-Len: 49.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.0% | KL: -0.6216
----------------------------------

Step 3210 | Loss: 0.7266 | Margin: 0.0713 | Acc: 62.5% | C-Len: 29.2 | R-Len: 54.0
Step 3220 | Loss: 0.4570 | Margin: 0.9766 | Acc: 75.0% | C-Len: 29.2 | R-Len: 59.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -0.7041
----------------------------------

Step 3230 | Loss: 0.7305 | Margin: 0.2168 | Acc: 50.0% | C-Len: 29.6 | R-Len: 23.0
Step 3240 | Loss: 0.3320 | Margin: 1.1406 | Acc: 100.0% | C-Len: 25.4 | R-Len: 22.9
Step 3250 | Loss: 0.6641 | Margin: 0.4277 | Acc: 75.0% | C-Len: 58.1 | R-Len: 44.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.0% | KL: -0.7748
----------------------------------

Step 3260 | Loss: 1.0781 | Margin: -0.5625 | Acc: 50.0% | C-Len: 22.1 | R-Len: 41.1
Step 3270 | Loss: 0.5508 | Margin: 0.4922 | Acc: 62.5% | C-Len: 60.5 | R-Len: 66.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -0.9545
----------------------------------

Step 3280 | Loss: 0.6562 | Margin: 0.2080 | Acc: 50.0% | C-Len: 29.1 | R-Len: 24.1
Step 3290 | Loss: 0.8203 | Margin: 0.1680 | Acc: 62.5% | C-Len: 24.9 | R-Len: 47.9
Step 3300 | Loss: 0.8125 | Margin: 0.1602 | Acc: 50.0% | C-Len: 58.6 | R-Len: 78.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -1.1186
----------------------------------

Step 3310 | Loss: 0.6953 | Margin: 0.3516 | Acc: 50.0% | C-Len: 20.9 | R-Len: 51.6
Step 3320 | Loss: 0.7188 | Margin: 0.2148 | Acc: 62.5% | C-Len: 40.6 | R-Len: 40.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 56.5% | KL: -1.0791
----------------------------------

Step 3330 | Loss: 0.8516 | Margin: 0.0591 | Acc: 62.5% | C-Len: 25.9 | R-Len: 76.6
Step 3340 | Loss: 0.6250 | Margin: 0.3750 | Acc: 50.0% | C-Len: 38.2 | R-Len: 48.2
Step 3350 | Loss: 0.6953 | Margin: 0.2363 | Acc: 62.5% | C-Len: 25.2 | R-Len: 46.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -1.1149
----------------------------------

Step 3360 | Loss: 0.6680 | Margin: 0.2598 | Acc: 62.5% | C-Len: 22.5 | R-Len: 27.9
Step 3370 | Loss: 0.7969 | Margin: 0.0850 | Acc: 50.0% | C-Len: 80.1 | R-Len: 91.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -1.2027
----------------------------------

Step 3380 | Loss: 0.5273 | Margin: 0.5352 | Acc: 75.0% | C-Len: 35.5 | R-Len: 42.1
Step 3390 | Loss: 0.5703 | Margin: 0.4648 | Acc: 62.5% | C-Len: 47.9 | R-Len: 37.5
Step 3400 | Loss: 0.5000 | Margin: 0.7148 | Acc: 75.0% | C-Len: 29.6 | R-Len: 33.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -1.3672
----------------------------------

Step 3410 | Loss: 0.6641 | Margin: 0.3984 | Acc: 50.0% | C-Len: 44.9 | R-Len: 65.4
Step 3420 | Loss: 0.9922 | Margin: -0.4473 | Acc: 37.5% | C-Len: 43.9 | R-Len: 34.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -1.4841
----------------------------------

Step 3430 | Loss: 0.4570 | Margin: 0.8945 | Acc: 87.5% | C-Len: 24.0 | R-Len: 36.9
Step 3440 | Loss: 0.5547 | Margin: 0.4141 | Acc: 62.5% | C-Len: 35.1 | R-Len: 27.0
Step 3450 | Loss: 0.9141 | Margin: -0.2754 | Acc: 25.0% | C-Len: 42.4 | R-Len: 58.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.0% | KL: -1.5569
----------------------------------

Step 3460 | Loss: 0.7070 | Margin: 0.1084 | Acc: 37.5% | C-Len: 61.1 | R-Len: 76.0
Step 3470 | Loss: 0.6016 | Margin: 0.4980 | Acc: 62.5% | C-Len: 26.2 | R-Len: 59.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -1.6192
----------------------------------

Step 3480 | Loss: 0.9141 | Margin: -0.2070 | Acc: 37.5% | C-Len: 50.9 | R-Len: 41.1
Step 3490 | Loss: 0.4902 | Margin: 0.8047 | Acc: 75.0% | C-Len: 14.2 | R-Len: 64.0
Step 3500 | Loss: 0.8164 | Margin: -0.1240 | Acc: 37.5% | C-Len: 58.2 | R-Len: 42.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -1.6163
----------------------------------

Step 3510 | Loss: 0.5117 | Margin: 0.6953 | Acc: 75.0% | C-Len: 26.8 | R-Len: 65.4
Step 3520 | Loss: 0.7969 | Margin: 0.1689 | Acc: 50.0% | C-Len: 62.5 | R-Len: 55.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -1.6712
----------------------------------

Step 3530 | Loss: 0.8164 | Margin: 0.0957 | Acc: 50.0% | C-Len: 69.4 | R-Len: 59.6
Step 3540 | Loss: 0.6133 | Margin: 0.4492 | Acc: 75.0% | C-Len: 49.6 | R-Len: 96.4
Step 3550 | Loss: 0.6562 | Margin: 0.2314 | Acc: 75.0% | C-Len: 52.5 | R-Len: 31.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 56.5% | KL: -1.6695
----------------------------------

Step 3560 | Loss: 0.9141 | Margin: -0.2852 | Acc: 37.5% | C-Len: 53.9 | R-Len: 41.1
Step 3570 | Loss: 0.7422 | Margin: 0.0605 | Acc: 37.5% | C-Len: 23.1 | R-Len: 48.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.0% | KL: -1.7520
----------------------------------

Step 3580 | Loss: 0.3926 | Margin: 1.0234 | Acc: 75.0% | C-Len: 40.9 | R-Len: 52.2
Step 3590 | Loss: 0.5664 | Margin: 0.5547 | Acc: 37.5% | C-Len: 43.8 | R-Len: 28.0
Step 3600 | Loss: 0.5469 | Margin: 0.5391 | Acc: 75.0% | C-Len: 41.6 | R-Len: 72.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -1.8662
----------------------------------

Step 3610 | Loss: 1.1016 | Margin: -0.4434 | Acc: 37.5% | C-Len: 37.6 | R-Len: 31.5
Step 3620 | Loss: 0.4180 | Margin: 0.8047 | Acc: 87.5% | C-Len: 27.4 | R-Len: 39.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -2.0261
----------------------------------

Step 3630 | Loss: 0.4727 | Margin: 0.6562 | Acc: 87.5% | C-Len: 49.0 | R-Len: 59.8
Step 3640 | Loss: 0.4668 | Margin: 0.5977 | Acc: 100.0% | C-Len: 33.1 | R-Len: 19.5
Step 3650 | Loss: 0.4668 | Margin: 0.8672 | Acc: 87.5% | C-Len: 23.1 | R-Len: 44.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -2.1475
----------------------------------

Step 3660 | Loss: 0.6289 | Margin: 0.2148 | Acc: 62.5% | C-Len: 19.8 | R-Len: 32.5
Step 3670 | Loss: 0.5039 | Margin: 0.5469 | Acc: 75.0% | C-Len: 37.5 | R-Len: 62.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -2.2115
----------------------------------

Step 3680 | Loss: 0.5977 | Margin: 0.4180 | Acc: 62.5% | C-Len: 52.1 | R-Len: 58.8
Step 3690 | Loss: 0.8281 | Margin: -0.0010 | Acc: 50.0% | C-Len: 28.2 | R-Len: 47.6
Step 3700 | Loss: 0.5977 | Margin: 0.4375 | Acc: 62.5% | C-Len: 26.0 | R-Len: 38.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -2.3383
----------------------------------

Step 3710 | Loss: 0.6914 | Margin: 0.2100 | Acc: 50.0% | C-Len: 49.8 | R-Len: 45.0
Step 3720 | Loss: 0.5898 | Margin: 0.3652 | Acc: 50.0% | C-Len: 29.0 | R-Len: 40.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.0% | KL: -2.4547
----------------------------------

Step 3730 | Loss: 0.7148 | Margin: 0.4004 | Acc: 62.5% | C-Len: 49.2 | R-Len: 42.9
Step 3740 | Loss: 0.5703 | Margin: 0.7656 | Acc: 62.5% | C-Len: 28.4 | R-Len: 51.6
Step 3750 | Loss: 0.5352 | Margin: 0.5156 | Acc: 62.5% | C-Len: 28.0 | R-Len: 60.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.5% | KL: -2.5261
----------------------------------

Step 3760 | Loss: 0.5586 | Margin: 0.7109 | Acc: 62.5% | C-Len: 31.9 | R-Len: 37.2
Step 3770 | Loss: 0.7695 | Margin: 0.3027 | Acc: 62.5% | C-Len: 29.2 | R-Len: 51.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -2.6437
----------------------------------

Step 3780 | Loss: 0.6133 | Margin: 0.2695 | Acc: 75.0% | C-Len: 37.2 | R-Len: 30.5
Step 3790 | Loss: 0.4023 | Margin: 0.8359 | Acc: 87.5% | C-Len: 33.4 | R-Len: 48.6
Step 3800 | Loss: 0.8242 | Margin: 0.2656 | Acc: 62.5% | C-Len: 35.8 | R-Len: 53.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -2.7249
----------------------------------

Step 3810 | Loss: 0.4922 | Margin: 0.8203 | Acc: 75.0% | C-Len: 49.9 | R-Len: 71.1
Step 3820 | Loss: 0.5469 | Margin: 0.8828 | Acc: 62.5% | C-Len: 65.8 | R-Len: 54.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.5% | KL: -2.8241
----------------------------------

Step 3830 | Loss: 0.6992 | Margin: 0.6953 | Acc: 25.0% | C-Len: 41.4 | R-Len: 78.5
Step 3840 | Loss: 0.5703 | Margin: 0.5078 | Acc: 75.0% | C-Len: 36.9 | R-Len: 35.2
Step 3850 | Loss: 1.0625 | Margin: -0.3184 | Acc: 50.0% | C-Len: 36.0 | R-Len: 59.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.9356
----------------------------------

Step 3860 | Loss: 0.5117 | Margin: 1.2422 | Acc: 75.0% | C-Len: 36.9 | R-Len: 64.2
Step 3870 | Loss: 0.4414 | Margin: 0.7969 | Acc: 87.5% | C-Len: 24.1 | R-Len: 63.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.5% | KL: -2.8885
----------------------------------

Step 3880 | Loss: 0.5781 | Margin: 0.5352 | Acc: 50.0% | C-Len: 21.0 | R-Len: 40.9
Step 3890 | Loss: 0.5664 | Margin: 0.5469 | Acc: 62.5% | C-Len: 29.1 | R-Len: 27.5
Step 3900 | Loss: 0.7578 | Margin: 0.5977 | Acc: 62.5% | C-Len: 27.6 | R-Len: 69.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.5% | KL: -2.8239
----------------------------------

Step 3910 | Loss: 0.4785 | Margin: 0.8867 | Acc: 75.0% | C-Len: 19.8 | R-Len: 41.4
Step 3920 | Loss: 0.3730 | Margin: 1.0938 | Acc: 87.5% | C-Len: 31.1 | R-Len: 57.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -2.7861
----------------------------------

Step 3930 | Loss: 0.3750 | Margin: 1.0625 | Acc: 87.5% | C-Len: 28.6 | R-Len: 45.1
Step 3940 | Loss: 0.4531 | Margin: 1.0703 | Acc: 87.5% | C-Len: 48.8 | R-Len: 78.2
Step 3950 | Loss: 0.2539 | Margin: 1.5781 | Acc: 100.0% | C-Len: 49.6 | R-Len: 59.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.5% | KL: -2.7591
----------------------------------

Step 3960 | Loss: 0.6094 | Margin: 0.4844 | Acc: 75.0% | C-Len: 46.6 | R-Len: 79.2
Step 3970 | Loss: 0.6641 | Margin: 0.4531 | Acc: 50.0% | C-Len: 31.8 | R-Len: 36.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.0% | KL: -2.8196
----------------------------------

Step 3980 | Loss: 0.8555 | Margin: -0.0859 | Acc: 37.5% | C-Len: 46.4 | R-Len: 56.4
Step 3990 | Loss: 0.6484 | Margin: 0.5547 | Acc: 62.5% | C-Len: 32.0 | R-Len: 44.4
Step 4000 | Loss: 0.5195 | Margin: 0.4668 | Acc: 75.0% | C-Len: 34.9 | R-Len: 50.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.5% | KL: -2.9581
----------------------------------

Step 4010 | Loss: 0.7031 | Margin: 0.5117 | Acc: 50.0% | C-Len: 40.9 | R-Len: 37.2
Step 4020 | Loss: 1.2422 | Margin: -0.5703 | Acc: 50.0% | C-Len: 34.6 | R-Len: 35.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -3.0416
----------------------------------

Step 4030 | Loss: 0.4160 | Margin: 1.0469 | Acc: 87.5% | C-Len: 64.0 | R-Len: 84.8
Step 4040 | Loss: 0.7461 | Margin: 0.0693 | Acc: 50.0% | C-Len: 29.9 | R-Len: 55.5
Step 4050 | Loss: 0.9219 | Margin: -0.1738 | Acc: 50.0% | C-Len: 31.5 | R-Len: 37.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.9858
----------------------------------

Step 4060 | Loss: 0.4375 | Margin: 1.3438 | Acc: 87.5% | C-Len: 30.0 | R-Len: 72.1
Step 4070 | Loss: 0.3379 | Margin: 1.3516 | Acc: 87.5% | C-Len: 18.9 | R-Len: 64.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.5% | KL: -2.9192
----------------------------------

Step 4080 | Loss: 1.0078 | Margin: -0.3691 | Acc: 37.5% | C-Len: 55.9 | R-Len: 45.5
Step 4090 | Loss: 0.6797 | Margin: 0.7422 | Acc: 62.5% | C-Len: 82.0 | R-Len: 82.6
Step 4100 | Loss: 0.6758 | Margin: 0.2266 | Acc: 62.5% | C-Len: 34.9 | R-Len: 25.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.5% | KL: -2.8940
----------------------------------

Step 4110 | Loss: 0.6875 | Margin: 0.2773 | Acc: 75.0% | C-Len: 35.1 | R-Len: 51.6
Step 4120 | Loss: 0.4258 | Margin: 1.2734 | Acc: 75.0% | C-Len: 32.2 | R-Len: 53.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.5% | KL: -2.8625
----------------------------------

Step 4130 | Loss: 0.6289 | Margin: 0.3887 | Acc: 50.0% | C-Len: 20.1 | R-Len: 40.4
Step 4140 | Loss: 0.6094 | Margin: 0.7617 | Acc: 62.5% | C-Len: 38.6 | R-Len: 52.6
Step 4150 | Loss: 0.3789 | Margin: 1.1250 | Acc: 87.5% | C-Len: 45.5 | R-Len: 40.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.8491
----------------------------------

Step 4160 | Loss: 0.5078 | Margin: 0.6797 | Acc: 50.0% | C-Len: 32.1 | R-Len: 43.2
Step 4170 | Loss: 0.7969 | Margin: -0.0776 | Acc: 50.0% | C-Len: 28.2 | R-Len: 65.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -2.9896
----------------------------------

Step 4180 | Loss: 0.5352 | Margin: 0.9727 | Acc: 62.5% | C-Len: 22.8 | R-Len: 36.9
Step 4190 | Loss: 0.6641 | Margin: 0.2314 | Acc: 50.0% | C-Len: 40.9 | R-Len: 63.0
Step 4200 | Loss: 0.7656 | Margin: 0.0339 | Acc: 50.0% | C-Len: 32.4 | R-Len: 59.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.5% | KL: -3.0382
----------------------------------

Step 4210 | Loss: 0.5469 | Margin: 0.4629 | Acc: 62.5% | C-Len: 39.1 | R-Len: 35.9
Step 4220 | Loss: 1.2500 | Margin: -0.6406 | Acc: 37.5% | C-Len: 103.8 | R-Len: 54.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.5% | KL: -2.9333
----------------------------------

Step 4230 | Loss: 0.9219 | Margin: -0.0251 | Acc: 62.5% | C-Len: 45.8 | R-Len: 55.5
Step 4240 | Loss: 0.5234 | Margin: 0.4922 | Acc: 62.5% | C-Len: 23.8 | R-Len: 51.2
Step 4250 | Loss: 0.7070 | Margin: 0.2344 | Acc: 62.5% | C-Len: 30.6 | R-Len: 43.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.5% | KL: -2.8526
----------------------------------

Step 4260 | Loss: 0.5547 | Margin: 0.7617 | Acc: 75.0% | C-Len: 51.9 | R-Len: 68.8
Step 4270 | Loss: 0.6523 | Margin: 0.4277 | Acc: 62.5% | C-Len: 18.6 | R-Len: 41.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -2.8125
----------------------------------

Step 4280 | Loss: 0.6562 | Margin: 0.3984 | Acc: 62.5% | C-Len: 19.9 | R-Len: 42.6
Step 4290 | Loss: 0.6953 | Margin: 0.3613 | Acc: 50.0% | C-Len: 45.6 | R-Len: 59.5
Step 4300 | Loss: 0.5742 | Margin: 0.4551 | Acc: 62.5% | C-Len: 36.0 | R-Len: 31.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.7905
----------------------------------

Step 4310 | Loss: 0.4883 | Margin: 0.6328 | Acc: 87.5% | C-Len: 23.8 | R-Len: 64.9
Step 4320 | Loss: 0.7031 | Margin: 0.1055 | Acc: 50.0% | C-Len: 20.8 | R-Len: 37.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 57.5% | KL: -2.7881
----------------------------------

Step 4330 | Loss: 0.7188 | Margin: -0.0027 | Acc: 50.0% | C-Len: 32.5 | R-Len: 39.8
Step 4340 | Loss: 0.4531 | Margin: 0.8633 | Acc: 87.5% | C-Len: 35.8 | R-Len: 61.1
Step 4350 | Loss: 0.9336 | Margin: 0.3438 | Acc: 50.0% | C-Len: 72.5 | R-Len: 64.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.7469
----------------------------------

Step 4360 | Loss: 0.5586 | Margin: 0.4238 | Acc: 62.5% | C-Len: 42.1 | R-Len: 44.5
Step 4370 | Loss: 0.4375 | Margin: 0.7773 | Acc: 87.5% | C-Len: 34.9 | R-Len: 49.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.5% | KL: -2.6993
----------------------------------

Step 4380 | Loss: 0.7695 | Margin: 0.1777 | Acc: 50.0% | C-Len: 45.5 | R-Len: 54.2
Step 4390 | Loss: 0.6406 | Margin: 0.4922 | Acc: 62.5% | C-Len: 34.9 | R-Len: 51.4
Step 4400 | Loss: 0.9922 | Margin: -0.3008 | Acc: 50.0% | C-Len: 31.8 | R-Len: 65.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -2.7477
----------------------------------

Step 4410 | Loss: 0.5430 | Margin: 0.5430 | Acc: 75.0% | C-Len: 50.9 | R-Len: 42.1
Step 4420 | Loss: 0.6406 | Margin: 0.5117 | Acc: 75.0% | C-Len: 22.0 | R-Len: 44.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.8132
----------------------------------

Step 4430 | Loss: 0.5547 | Margin: 0.3691 | Acc: 62.5% | C-Len: 25.4 | R-Len: 41.2
Step 4440 | Loss: 0.5156 | Margin: 0.5273 | Acc: 87.5% | C-Len: 53.9 | R-Len: 43.6
Step 4450 | Loss: 0.5703 | Margin: 0.5469 | Acc: 62.5% | C-Len: 47.1 | R-Len: 66.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.5% | KL: -2.8565
----------------------------------

Step 4460 | Loss: 0.4414 | Margin: 0.9453 | Acc: 75.0% | C-Len: 32.2 | R-Len: 41.9
Step 4470 | Loss: 0.5430 | Margin: 0.4922 | Acc: 62.5% | C-Len: 23.0 | R-Len: 36.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.5% | KL: -2.8353
----------------------------------

Step 4480 | Loss: 0.6797 | Margin: 0.2969 | Acc: 62.5% | C-Len: 35.2 | R-Len: 51.1
Step 4490 | Loss: 0.6641 | Margin: 0.2695 | Acc: 62.5% | C-Len: 52.2 | R-Len: 45.1
Step 4500 | Loss: 0.6250 | Margin: 0.2256 | Acc: 62.5% | C-Len: 33.4 | R-Len: 23.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.8108
----------------------------------

Step 4510 | Loss: 0.5703 | Margin: 0.5039 | Acc: 75.0% | C-Len: 14.2 | R-Len: 41.4
Step 4520 | Loss: 0.6797 | Margin: 0.1396 | Acc: 62.5% | C-Len: 27.1 | R-Len: 43.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.9167
----------------------------------

Step 4530 | Loss: 0.4961 | Margin: 0.9453 | Acc: 62.5% | C-Len: 29.9 | R-Len: 47.5
Step 4540 | Loss: 0.4492 | Margin: 0.6953 | Acc: 87.5% | C-Len: 23.0 | R-Len: 68.0
Step 4550 | Loss: 0.6289 | Margin: 0.3887 | Acc: 62.5% | C-Len: 38.8 | R-Len: 41.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.5% | KL: -2.8745
----------------------------------

Step 4560 | Loss: 1.0781 | Margin: -0.3066 | Acc: 37.5% | C-Len: 53.8 | R-Len: 46.1
Step 4570 | Loss: 0.5625 | Margin: 0.4297 | Acc: 75.0% | C-Len: 41.6 | R-Len: 65.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.5% | KL: -2.7852
----------------------------------

Step 4580 | Loss: 0.4902 | Margin: 0.8984 | Acc: 50.0% | C-Len: 28.8 | R-Len: 68.1
Step 4590 | Loss: 0.6055 | Margin: 0.2930 | Acc: 62.5% | C-Len: 29.6 | R-Len: 20.6
Step 4600 | Loss: 0.5703 | Margin: 0.4219 | Acc: 75.0% | C-Len: 37.9 | R-Len: 31.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.0% | KL: -2.7278
----------------------------------

Step 4610 | Loss: 0.5391 | Margin: 0.9258 | Acc: 75.0% | C-Len: 48.4 | R-Len: 50.9
Step 4620 | Loss: 0.4961 | Margin: 0.6992 | Acc: 75.0% | C-Len: 56.1 | R-Len: 56.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.5% | KL: -2.6258
----------------------------------

Step 4630 | Loss: 0.6094 | Margin: 0.5000 | Acc: 75.0% | C-Len: 36.5 | R-Len: 50.0
Step 4640 | Loss: 0.7070 | Margin: 0.2812 | Acc: 50.0% | C-Len: 28.0 | R-Len: 28.6
Step 4650 | Loss: 1.0156 | Margin: -0.4609 | Acc: 25.0% | C-Len: 45.5 | R-Len: 56.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.5505
----------------------------------

Step 4660 | Loss: 0.5430 | Margin: 0.9414 | Acc: 62.5% | C-Len: 31.9 | R-Len: 47.1
Step 4670 | Loss: 0.5508 | Margin: 0.5234 | Acc: 62.5% | C-Len: 29.5 | R-Len: 37.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.0% | KL: -2.5204
----------------------------------

Step 4680 | Loss: 0.6133 | Margin: 0.3008 | Acc: 75.0% | C-Len: 39.6 | R-Len: 63.8
Step 4690 | Loss: 0.3359 | Margin: 1.4453 | Acc: 75.0% | C-Len: 13.8 | R-Len: 50.2
Step 4700 | Loss: 0.7031 | Margin: 0.1157 | Acc: 62.5% | C-Len: 42.6 | R-Len: 26.0

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.6165
----------------------------------

Step 4710 | Loss: 0.4727 | Margin: 0.6445 | Acc: 87.5% | C-Len: 46.8 | R-Len: 71.1
Step 4720 | Loss: 0.8203 | Margin: -0.0610 | Acc: 50.0% | C-Len: 49.8 | R-Len: 67.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.6254
----------------------------------

Step 4730 | Loss: 0.8164 | Margin: 0.6562 | Acc: 50.0% | C-Len: 37.4 | R-Len: 55.0
Step 4740 | Loss: 0.8008 | Margin: -0.0547 | Acc: 75.0% | C-Len: 23.9 | R-Len: 62.5
Step 4750 | Loss: 0.3906 | Margin: 1.2422 | Acc: 87.5% | C-Len: 27.8 | R-Len: 41.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 58.0% | KL: -2.7518
----------------------------------

Step 4760 | Loss: 0.8828 | Margin: 0.0947 | Acc: 37.5% | C-Len: 34.6 | R-Len: 35.9
Step 4770 | Loss: 0.8906 | Margin: 0.0330 | Acc: 50.0% | C-Len: 25.9 | R-Len: 44.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.5% | KL: -2.8173
----------------------------------

Step 4780 | Loss: 0.7539 | Margin: 0.3301 | Acc: 37.5% | C-Len: 51.1 | R-Len: 43.2
Step 4790 | Loss: 0.5898 | Margin: 0.4219 | Acc: 62.5% | C-Len: 29.9 | R-Len: 41.9
Step 4800 | Loss: 0.9375 | Margin: 0.0229 | Acc: 50.0% | C-Len: 43.1 | R-Len: 61.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.5% | KL: -2.7330
----------------------------------

Step 4810 | Loss: 1.1016 | Margin: -0.0806 | Acc: 50.0% | C-Len: 70.6 | R-Len: 67.8
Step 4820 | Loss: 0.5586 | Margin: 0.6602 | Acc: 62.5% | C-Len: 22.0 | R-Len: 46.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.5% | KL: -2.7217
----------------------------------

Step 4830 | Loss: 0.4453 | Margin: 0.6094 | Acc: 100.0% | C-Len: 30.1 | R-Len: 29.0
Step 4840 | Loss: 0.4277 | Margin: 0.9336 | Acc: 87.5% | C-Len: 21.8 | R-Len: 53.0
Step 4850 | Loss: 0.5781 | Margin: 0.5312 | Acc: 62.5% | C-Len: 36.9 | R-Len: 42.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 59.0% | KL: -2.6848
----------------------------------

Step 4860 | Loss: 0.9062 | Margin: 0.1709 | Acc: 37.5% | C-Len: 45.8 | R-Len: 69.5
Step 4870 | Loss: 0.6641 | Margin: 0.5469 | Acc: 50.0% | C-Len: 47.6 | R-Len: 97.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.0% | KL: -2.7259
----------------------------------

Step 4880 | Loss: 0.5352 | Margin: 0.5859 | Acc: 50.0% | C-Len: 53.5 | R-Len: 52.0
Step 4890 | Loss: 0.7188 | Margin: 0.5156 | Acc: 62.5% | C-Len: 32.8 | R-Len: 30.0
Step 4900 | Loss: 0.8086 | Margin: -0.0559 | Acc: 50.0% | C-Len: 37.4 | R-Len: 55.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.5% | KL: -2.6146
----------------------------------

Step 4910 | Loss: 0.4180 | Margin: 0.8438 | Acc: 75.0% | C-Len: 30.4 | R-Len: 87.4
Step 4920 | Loss: 0.8203 | Margin: 0.4102 | Acc: 62.5% | C-Len: 44.1 | R-Len: 48.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 61.5% | KL: -2.4954
----------------------------------

Step 4930 | Loss: 0.8516 | Margin: -0.1504 | Acc: 37.5% | C-Len: 45.6 | R-Len: 31.9
Step 4940 | Loss: 0.7344 | Margin: 0.2090 | Acc: 62.5% | C-Len: 37.4 | R-Len: 64.2
Step 4950 | Loss: 0.5312 | Margin: 0.4863 | Acc: 87.5% | C-Len: 21.2 | R-Len: 43.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 61.0% | KL: -2.3283
----------------------------------

Step 4960 | Loss: 0.6680 | Margin: 0.1504 | Acc: 50.0% | C-Len: 52.8 | R-Len: 49.1
Step 4970 | Loss: 0.4727 | Margin: 0.7656 | Acc: 75.0% | C-Len: 42.1 | R-Len: 21.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.5% | KL: -2.2668
----------------------------------

Step 4980 | Loss: 0.6758 | Margin: 0.3633 | Acc: 87.5% | C-Len: 45.8 | R-Len: 59.9
Step 4990 | Loss: 0.4512 | Margin: 0.7227 | Acc: 75.0% | C-Len: 31.9 | R-Len: 53.4
Step 5000 | Loss: 0.3125 | Margin: 1.3281 | Acc: 87.5% | C-Len: 34.9 | R-Len: 26.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.5% | KL: -2.2299
----------------------------------

Step 5010 | Loss: 0.6445 | Margin: 0.5508 | Acc: 75.0% | C-Len: 36.9 | R-Len: 26.1
Step 5020 | Loss: 0.6523 | Margin: 0.7031 | Acc: 75.0% | C-Len: 25.5 | R-Len: 29.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 61.5% | KL: -2.1454
----------------------------------

Step 5030 | Loss: 0.7148 | Margin: 0.2354 | Acc: 62.5% | C-Len: 35.4 | R-Len: 23.9
Step 5040 | Loss: 0.7695 | Margin: 0.1064 | Acc: 37.5% | C-Len: 54.8 | R-Len: 29.8
Step 5050 | Loss: 0.3457 | Margin: 0.9805 | Acc: 100.0% | C-Len: 61.2 | R-Len: 54.1

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.0% | KL: -2.0714
----------------------------------

Step 5060 | Loss: 0.5312 | Margin: 0.4375 | Acc: 87.5% | C-Len: 40.5 | R-Len: 29.5
Step 5070 | Loss: 0.6719 | Margin: 0.2295 | Acc: 50.0% | C-Len: 36.8 | R-Len: 27.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.5% | KL: -2.0210
----------------------------------

Step 5080 | Loss: 0.6445 | Margin: 0.1660 | Acc: 50.0% | C-Len: 61.5 | R-Len: 72.2
Step 5090 | Loss: 0.4980 | Margin: 0.6719 | Acc: 75.0% | C-Len: 39.5 | R-Len: 54.4
Step 5100 | Loss: 0.5469 | Margin: 0.5430 | Acc: 87.5% | C-Len: 24.6 | R-Len: 31.4

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.5% | KL: -1.9992
----------------------------------

Step 5110 | Loss: 0.6484 | Margin: 0.2988 | Acc: 62.5% | C-Len: 40.6 | R-Len: 37.1
Step 5120 | Loss: 0.4551 | Margin: 0.9961 | Acc: 75.0% | C-Len: 36.8 | R-Len: 43.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 61.0% | KL: -2.0349
----------------------------------

Step 5130 | Loss: 0.8359 | Margin: -0.1035 | Acc: 37.5% | C-Len: 12.5 | R-Len: 19.9
Step 5140 | Loss: 0.4316 | Margin: 0.9023 | Acc: 75.0% | C-Len: 34.1 | R-Len: 45.2
Step 5150 | Loss: 0.5234 | Margin: 0.5039 | Acc: 75.0% | C-Len: 57.9 | R-Len: 46.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 61.5% | KL: -2.0590
----------------------------------

Step 5160 | Loss: 0.7148 | Margin: 0.3633 | Acc: 62.5% | C-Len: 35.0 | R-Len: 41.0
Step 5170 | Loss: 0.6875 | Margin: 0.5391 | Acc: 50.0% | C-Len: 43.9 | R-Len: 57.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 61.5% | KL: -2.1670
----------------------------------

Step 5180 | Loss: 0.6836 | Margin: 0.1895 | Acc: 37.5% | C-Len: 23.0 | R-Len: 30.0
Step 5190 | Loss: 0.8672 | Margin: 0.1914 | Acc: 75.0% | C-Len: 41.8 | R-Len: 46.9
Step 5200 | Loss: 0.5117 | Margin: 0.9336 | Acc: 75.0% | C-Len: 53.0 | R-Len: 23.6

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.5% | KL: -2.1967
----------------------------------

Step 5210 | Loss: 0.6914 | Margin: 0.1260 | Acc: 62.5% | C-Len: 38.1 | R-Len: 26.9
Step 5220 | Loss: 0.7500 | Margin: 0.2539 | Acc: 62.5% | C-Len: 35.1 | R-Len: 43.5

--- Running 25-Step Evaluation ---
EVAL | Acc: 61.0% | KL: -2.2753
----------------------------------

Step 5230 | Loss: 0.7500 | Margin: 0.0199 | Acc: 37.5% | C-Len: 41.6 | R-Len: 53.4
Step 5240 | Loss: 0.6172 | Margin: 0.3965 | Acc: 62.5% | C-Len: 52.8 | R-Len: 60.4
Step 5250 | Loss: 0.4199 | Margin: 0.7656 | Acc: 87.5% | C-Len: 63.8 | R-Len: 59.8

--- Running 25-Step Evaluation ---
EVAL | Acc: 62.5% | KL: -2.3891
----------------------------------

Step 5260 | Loss: 0.8125 | Margin: -0.0127 | Acc: 37.5% | C-Len: 43.6 | R-Len: 48.0
Step 5270 | Loss: 0.8203 | Margin: -0.0874 | Acc: 62.5% | C-Len: 45.5 | R-Len: 49.9

--- Running 25-Step Evaluation ---
EVAL | Acc: 60.5% | KL: -2.4450
----------------------------------

Step 5280 | Loss: 0.4609 | Margin: 0.6133 | Acc: 87.5% | C-Len: 21.0 | R-Len: 29.5
Step 5290 | Loss: 0.4941 | Margin: 1.0547 | Acc: 62.5% | C-Len: 27.8 | R-Len: 56.1
Step 5300 | Loss: 0.7188 | Margin: 0.0596 | Acc: 25.0% | C-Len: 50.2 | R-Len: 48.2

--- Running 25-Step Evaluation ---
EVAL | Acc: 62.0% | KL: -2.4839
----------------------------------

Step 5310 | Loss: 0.5938 | Margin: 0.3145 | Acc: 75.0% | C-Len: 22.0 | R-Len: 33.8
"""

# Regex patterns
step_pattern = re.compile(r"Step (\d+) \| Loss: ([\d.-]+) \| Margin: ([\d.-]+) \| Acc: ([\d.-]+)% \| C-Len: ([\d.-]+) \| R-Len: ([\d.-]+)")
eval_pattern = re.compile(r"--- Running 25-Step Evaluation ---\nEVAL \| Acc: ([\d.-]+)% \| KL: ([\d.-]+)")

steps, losses, margins, accs, c_lens, r_lens = [], [], [], [], [], []
eval_accs, eval_kls = [], []

for match in step_pattern.finditer(raw_logs):
    steps.append(int(match.group(1)))
    losses.append(float(match.group(2)))
    margins.append(float(match.group(3)))
    accs.append(float(match.group(4)))
    c_lens.append(float(match.group(5)))
    r_lens.append(float(match.group(6)))

for match in eval_pattern.finditer(raw_logs):
    eval_accs.append(float(match.group(1)))
    eval_kls.append(float(match.group(2)))

# Eval steps happen every 25 steps starting at step 25
eval_steps = [25 * (i + 1) for i in range(len(eval_accs))]

fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Full DPO Training Telemetry (1 Epoch)', fontsize=16, fontweight='bold')

axs[0, 0].plot(steps, accs, color='blue', alpha=0.3, label='Batch Acc')
# Calculate a rolling average for smoother visualization
rolling_acc = [sum(accs[max(0, i-10):i+1])/len(accs[max(0, i-10):i+1]) for i in range(len(accs))]
axs[0, 0].plot(steps, rolling_acc, color='darkblue', linewidth=2, label='Trend')
axs[0, 0].axhline(50, color='gray', linestyle='--')
axs[0, 0].set_title('Training Preference Accuracy')
axs[0, 0].legend()

axs[0, 1].plot(steps, margins, color='purple', alpha=0.5)
axs[0, 1].axhline(0, color='gray', linestyle='--')
axs[0, 1].set_title('Implicit Reward Margin ($z$)')

axs[1, 0].plot(steps, c_lens, color='green', alpha=0.6, label='Chosen Length')
axs[1, 0].plot(steps, r_lens, color='red', alpha=0.6, label='Rejected Length')
axs[1, 0].set_title('Verbosity Bias Tracker')
axs[1, 0].legend()

axs[1, 1].plot(eval_steps, eval_accs, color='darkorange', linewidth=2)
axs[1, 1].set_title('Held-Out Eval Accuracy')
axs[1, 1].set_xlabel('Global Step')

plt.tight_layout()
plt.show()

### Task C5: GRPO Implementation

In [ ]:
# CELL 15: GSM8K PIPELINE & VERIFIABLE REWARD (Task C6.1)
import re
from datasets import load_dataset

print("Loading GSM8K Dataset...")
gsm8k = load_dataset("openai/gsm8k", "main")
train_gsm8k = gsm8k['train']
test_gsm8k = gsm8k['test']

# 1. The Answer Extractor
def extract_answer(text):
    """
    Parses the final numeric answer from a solution string.
    Handles commas, decimals, negative numbers, and common LLM output formats.
    """
    if not isinstance(text, str):
        return None

    # Check for the official GSM8K ground truth format first
    if "####" in text:
        ans_str = text.split("####")[-1].strip()
        # Remove commas for safe float conversion
        ans_str = ans_str.replace(",", "")
        try:
            return float(ans_str)
        except ValueError:
            pass

    # Regex to find numbers (including negative, decimals, and comma-formatted)
    # This looks for the LAST number in the text, assuming it's the final answer.
    numbers = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text)
    if numbers:
        # Take the very last number found in the string
        final_num_str = numbers[-1].replace(",", "")
        try:
            return float(final_num_str)
        except ValueError:
            return None

    return None

# 2. The Verifiable Reward Function (r_v)
def verifiable_reward(generated_solution, gold_solution):
    """
    Returns 1 if the extracted generated answer matches the gold answer, else 0.
    """
    gen_ans = extract_answer(generated_solution)
    gold_ans = extract_answer(gold_solution)

    # If the extractor completely failed to find a number, reward is 0
    if gen_ans is None or gold_ans is None:
        return 0.0

    # Check if they match (using a tiny tolerance for float precision issues)
    if abs(gen_ans - gold_ans) < 1e-5:
        return 1.0
    return 0.0

# 3. Sanity Checks (Task C6.1 Step 4)
print("\nRunning Extractor Sanity Checks...")
success_count = 0
fail_count = 0

# Check 20 Gold Solutions (Should all be 1.0)
for i in range(20):
    gold_text = train_gsm8k[i]['answer']
    # We pass the gold text as BOTH generated and gold to test a perfect match
    r = verifiable_reward(gold_text, gold_text)
    if r == 1.0: success_count += 1

# Check 20 Obviously Wrong Solutions (Should all be 0.0)
wrong_texts = [
    "The answer is clearly 99999.",
    "So, if we add them up, we get 0.5.",
    "I'm not sure, maybe -42?",
    "No numbers here!",
    "The final result is 1,234.56"
] * 4 # Duplicate to make 20

for i in range(20):
    gold_text = train_gsm8k[i]['answer']
    r = verifiable_reward(wrong_texts[i], gold_text)
    if r == 0.0: fail_count += 1

print(f"Gold Match Test: {success_count}/20 passed (Expected 20/20)")
print(f"Wrong Match Test: {fail_count}/20 passed (Expected 20/20)")

if success_count == 20 and fail_count == 20:
    print("✅ Verifiable Reward (r_v) is mathematically sound and ready for GRPO.")
else:
    print("❌ Extractor failed. Review regex logic.")

In [ ]:
# OPTIONAL CELL: SAVE DATASETS TO DISK
import os

print("Saving processed datasets to disk...")
os.makedirs("./saved_datasets", exist_ok=True)

if 'parsed_train' in globals():
    parsed_train.save_to_disk("./saved_datasets/hh_rlhf_train")
if 'parsed_test' in globals():
    parsed_test.save_to_disk("./saved_datasets/hh_rlhf_test")
if 'train_gsm8k' in globals():
    train_gsm8k.save_to_disk("./saved_datasets/gsm8k_train")
if 'test_gsm8k' in globals():
    test_gsm8k.save_to_disk("./saved_datasets/gsm8k_test")

print("Data state secured. You can safely shut down the kernel.")

In [ ]:
# OPTIONAL CELL: BACKUP TO GOOGLE DRIVE
import os

# 1. Define the backup folder in your Google Drive
drive_backup_path = "/content/drive/MyDrive/LLM_Alignment_PA2_Backup"
os.makedirs(drive_backup_path, exist_ok=True)

print("Copying checkpoints to Google Drive...")

# 2. Use bash commands to recursively copy the folders
!cp -r /content/rm_checkpoint {drive_backup_path}/
!cp -r /content/sft_checkpoint {drive_backup_path}/
!cp -r /content/dpo_checkpoint {drive_backup_path}/

# 3. Copy the datasets if you ran the previous serialization cell
if os.path.exists("/content/saved_datasets"):
    print("Copying parsed datasets to Google Drive...")
    !cp -r /content/saved_datasets {drive_backup_path}/

print(f"\nAll artifacts successfully backed up to: {drive_backup_path}")
print("You are officially safe to disconnect and close the Colab tab!")

In [ ]:
# 3. Copy the datasets if you ran the previous serialization cell
if os.path.exists("/content/saved_datasets"):
    print("Copying parsed datasets to Google Drive...")
    !cp -r /content/saved_datasets {drive_backup_path}/

# State Revivial

In [ ]:
# CELL 15: RESTORE SESSION & RLVR SETUP (Tasks C5/C6)
import os
import re
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# --- 1. MOUNT DRIVE & SET PATHS ---
from google.colab import drive
drive.mount('/content/drive')

drive_backup_path = "/content/drive/MyDrive/LLM_Alignment_PA2_Backup"
sft_checkpoint_path = f"{drive_backup_path}/sft_checkpoint"

policy_model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# --- 2. LOAD SFT POLICY (pi_theta) ---
print("\nLoading Base Policy...")
policy_tokenizer = AutoTokenizer.from_pretrained(policy_model_id)
# Ensure left padding for generation
policy_tokenizer.padding_side = "left"
if policy_tokenizer.pad_token is None:
    policy_tokenizer.pad_token = policy_tokenizer.eos_token

policy_base = AutoModelForCausalLM.from_pretrained(
    policy_model_id, torch_dtype=torch.bfloat16, device_map="auto"
)
policy_base.gradient_checkpointing_enable()

print("Applying SFT Adapters from Drive...")
# Load as trainable so we can optimize it via GRPO
policy_model = PeftModel.from_pretrained(policy_base, sft_checkpoint_path, is_trainable=True)

# --- 3. LOAD GSM8K DATASET ---
print("\nLoading GSM8K Math Dataset...")
gsm8k = load_dataset("openai/gsm8k", "main")
train_gsm8k = gsm8k['train']
test_gsm8k = gsm8k['test']

# --- 4. THE VERIFIABLE REWARD ENGINE (Task C6.1) ---
def extract_answer(text):
    """Parses the final numeric answer from an LLM response or Gold solution."""
    if not isinstance(text, str): return None

    # Check for gold format first
    if "####" in text:
        ans_str = text.split("####")[-1].strip().replace(",", "")
        try: return float(ans_str)
        except ValueError: pass

    # Regex to find the last number in the string
    numbers = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text)
    if numbers:
        try: return float(numbers[-1].replace(",", ""))
        except ValueError: return None
    return None

def verifiable_reward(generated_solution, gold_solution):
    """Returns 1.0 if answers match exactly, else 0.0"""
    gen_ans = extract_answer(generated_solution)
    gold_ans = extract_answer(gold_solution)
    if gen_ans is None or gold_ans is None: return 0.0
    if abs(gen_ans - gold_ans) < 1e-5: return 1.0
    return 0.0

# --- 5. SANITY CHECKS ---
print("\nRunning Verifiable Reward Sanity Checks...")
success_count, fail_count = 0, 0
for i in range(20):
    gold = train_gsm8k[i]['answer']
    if verifiable_reward(gold, gold) == 1.0: success_count += 1
    if verifiable_reward("The answer is fish.", gold) == 0.0: fail_count += 1

print(f"Extractor Math Checks: {success_count}/20 passed.")
print(f"Extractor Fail Checks: {fail_count}/20 passed.")
if success_count == 20 and fail_count == 20:
    print("✅ Session Restored. RLVR Engine Online.")
else:
    print("❌ Extractor failed check.")

# Tasks C5 and C6: GRPO and RLVR

In [ ]:
# CELL 16: GRPO & RLVR TRAINING ENGINE (Fully Compliant)
import torch
import torch.nn.functional as F
from torch.optim import AdamW
import numpy as np

# 1. Hyperparameters (C6.3 Step 10)
update_steps = 300
batch_size = 8
K = 4
max_new_tokens = 256 # Note: Watch for OOM. Reduce to 128 if Colab crashes.
beta = 0.05
epsilon = 0.2

optimizer = AdamW(filter(lambda p: p.requires_grad, policy_model.parameters()), lr=5e-6)

# 2. Prompt Formatter (C6.2 Steps 5 & 6)
def format_gsm8k_prompt(question):
    q_tokens = policy_tokenizer.encode(question)
    if len(q_tokens) > 200:
        question = policy_tokenizer.decode(q_tokens[:200])
    return f"Solve the following math problem step by step.\nAt the end, write your final answer as a single number.\nProblem: {question}\nSolution:\n"

def get_per_token_logprobs(logits, input_ids):
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = input_ids[..., 1:].contiguous()
    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction='none')
    return -loss.view(shift_labels.size())

print("Starting RLVR Training with GRPO...")
step = 0
train_iter = iter(train_gsm8k)

for step in range(1, update_steps + 1):
    policy_model.train()

    # Prepare batch
    prompts, gold_answers = [], []
    for _ in range(batch_size):
        try: example = next(train_iter)
        except StopIteration:
            train_iter = iter(train_gsm8k)
            example = next(train_iter)
        prompts.append(format_gsm8k_prompt(example['question']))
        gold_answers.append(example['answer'])

    # ==========================================
    # C5.1: GROUP ROLLOUT & ADVANTAGE
    # ==========================================
    policy_model.eval()
    batch_trajectories = []
    degenerate_count = 0
    group_rewards_log, response_lengths = [], []
    format_compliance_count = 0 # C6.4 Step 12

    with torch.no_grad():
        for b_idx in range(batch_size):
            prompt, gold = prompts[b_idx], gold_answers[b_idx]

            inputs = policy_tokenizer([prompt] * K, return_tensors="pt", padding=True).to("cuda")
            prompt_len = inputs['input_ids'].shape[1]

            gen_outputs = policy_model.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=0.8,
                do_sample=True, pad_token_id=policy_tokenizer.pad_token_id
            )

            rewards = []
            for k in range(K):
                response_ids = gen_outputs[k][prompt_len:]
                response_text = policy_tokenizer.decode(response_ids, skip_special_tokens=True)

                # Format Compliance Check
                if extract_answer(response_text) is not None:
                    format_compliance_count += 1

                rewards.append(verifiable_reward(response_text, gold))
                response_lengths.append(len(response_ids))

            rewards = torch.tensor(rewards).to("cuda")
            group_mean = rewards.mean()
            group_rewards_log.append(group_mean.item())

            if rewards.std() == 0:
                degenerate_count += 1

            advantages = rewards - group_mean # C5.1 Step 1

            for k in range(K):
                full_ids = gen_outputs[k].unsqueeze(0)
                resp_len = len(gen_outputs[k]) - prompt_len

                old_logits = policy_model(full_ids).logits
                old_logprobs = get_per_token_logprobs(old_logits, full_ids)[0, -resp_len:]

                with policy_model.disable_adapter():
                    ref_logits = policy_model(full_ids).logits
                    ref_logprobs = get_per_token_logprobs(ref_logits, full_ids)[0, -resp_len:]

                batch_trajectories.append({
                    "ids": full_ids, "resp_len": resp_len,
                    "old_logprobs": old_logprobs, "ref_logprobs": ref_logprobs,
                    "advantage": advantages[k]
                })

    # C5.1 Step 2: Standardize Advantages
    all_advs = torch.stack([t["advantage"] for t in batch_trajectories])
    adv_mean = all_advs.mean()
    adv_std = all_advs.std() + 1e-8

    # C6.4 Step 14: Credit Assignment Fraction
    # Tokens get gradient if they belong to a sequence with a non-zero advantage
    nonzero_grad_fraction = (all_advs != 0).float().mean().item()

    # ==========================================
    # C5.2: GRPO UPDATE
    # ==========================================
    policy_model.train()
    optimizer.zero_grad()
    step_loss, step_kl = 0, 0

    for traj in batch_trajectories:
        ids, resp_len, adv = traj["ids"], traj["resp_len"], traj["advantage"]
        norm_adv = (adv - adv_mean) / adv_std

        logits = policy_model(ids).logits
        current_logprobs = get_per_token_logprobs(logits, ids)[0, -resp_len:]

        ratio = torch.exp(current_logprobs - traj["old_logprobs"].detach())

        # C5.2 Step 6: 1/T_k normalization (resp_len)
        surr1 = ratio * norm_adv
        surr2 = torch.clamp(ratio, 1.0 - epsilon, 1.0 + epsilon) * norm_adv
        l_clip = -torch.min(surr1, surr2).sum() / resp_len

        kl_div = (current_logprobs - traj["ref_logprobs"].detach()).sum() / resp_len
        loss = (l_clip + beta * kl_div) / (batch_size * K)

        loss.backward()
        step_loss += loss.item()
        step_kl += kl_div.item() / (batch_size * K)

    optimizer.step()

    # ==========================================
    # C6.3 / C6.4: EVALUATION & TELEMETRY
    # ==========================================
    if step % 25 == 0 or step == 1:
        deg_frac = degenerate_count / batch_size
        format_rate = format_compliance_count / (batch_size * K)
        print(f"Step {step:03d} | Train R_v: {np.mean(group_rewards_log):.3f} | Degen: {deg_frac*100:.1f}% | Format OK: {format_rate*100:.1f}% | GradTokens: {nonzero_grad_fraction*100:.1f}% | KL: {step_kl:.4f}")

    # Formal pass@1 Evaluation (Every 25 steps)
    if step > 0 and step % 25 == 0:
        print(f"\n--- Running pass@1 Evaluation on Test Set ---")
        policy_model.eval()
        eval_pass_count = 0
        eval_samples = 50 # Evaluate on 50 to save time (adjust as needed)

        with torch.no_grad():
            for i in range(eval_samples):
                e_prompt = format_gsm8k_prompt(test_gsm8k[i]['question'])
                e_gold = test_gsm8k[i]['answer']

                e_inputs = policy_tokenizer(e_prompt, return_tensors="pt").to("cuda")
                e_len = e_inputs['input_ids'].shape[1]

                # Greedy decoding for evaluation (do_sample=False)
                e_gen = policy_model.generate(**e_inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=policy_tokenizer.pad_token_id)
                e_text = policy_tokenizer.decode(e_gen[0][e_len:], skip_special_tokens=True)

                if verifiable_reward(e_text, e_gold) == 1.0:
                    eval_pass_count += 1

        pass_at_1 = eval_pass_count / eval_samples
        print(f"EVAL | pass@1: {pass_at_1*100:.1f}% ({eval_pass_count}/{eval_samples})\n")

print("\nSaving RLVR Checkpoint...")
policy_model.save_pretrained("./rlvr_checkpoint")
print("RLVR Training Complete.")